# Eksperimen 21: Temporal Window Alignment & Inductive Setup (SHINE FRAUD II)

Notebook ini mengimplementasikan eksperimen `EXP21`:
- Partisi rolling window 2016-2020: 300 hari Train, 30 hari Val, 8 hari Embargo Gap, 60 hari Test.
- Mengeliminasi 30-year concept drift.
- Ekstraksi fitur GraphSAGE + XGBoost Classifier.
- Kalibrasi threshold dinamis tau* pada Validation Set.


In [ ]:
%%writefile config.kaggle.yaml
data:
  transactions: "/kaggle/input/credit-card-transactions/credit_card_transactions-ibm_v2.csv"
  max_rows: null
  start_year: 2016

time_window:
  reference_date: "2018-01-01" # Tanggal T0 (era modern 2016-2020 dengan sebaran fraud stabil)
  train_days: 300
  val_days: 30
  gap_days: 8
  test_days: 60

sampling:
  method: "uniform" # Pilihan: uniform, topology, importance
  undersampling_ratio: 1.0 # 1.0 = tidak ada undersampling (mempertahankan topologi asli)

model:
  hidden_channels: 256 # Sesuai gambar: Hidden Dimension 256 (ReLU & BatchNorm)
  out_channels: 1
  num_layers: 2 # Sesuai gambar: Lapisan GraphSAGE 2
  aggr: "mean" # Sesuai gambar: Mean Aggregator
  dropout: 0.2 # Regularisasi dropout optimal

training:
  batch_size: 1024 # Sesuai gambar: Ukuran Batch 1024
  epochs: 100 # Maksimal 100 epoch dengan early stopping
  learning_rate: 0.001 # Sesuai gambar: Learning Rate 0.001 (Adam)
  weight_decay: 0.0001 # Regularisasi L2 seimbang pada Adam optimizer
  label_smoothing: 0.0 # Dinonaktifkan: Mencegah tabrakan matematis dengan pos_weight & mode collapse
  grad_clip_norm: 5.0 # Mencegah exploding gradient tanpa memotong sinyal fraud alami
  warmup_epochs: 0 # Dinonaktifkan: Pembelajaran aktif sejak epoch pertama
  use_lr_scheduler: true # Kurangi LR saat performa validasi mendatar
  lr_scheduler_factor: 0.5 # Sesuai gambar: Decay 0.5x
  lr_scheduler_patience: 5 # Sesuai gambar: Decay per 5 epoch
  early_stopping_patience: 15 # Sesuai gambar: Patience 15 epoch tanpa peningkatan AUPRC
  loss_weight_strategy: "sqrt" # Pilihan: sqrt (teredam ~27x), clipped (maks batas), full (rasio penuh ~732x)
  max_pos_weight: 50.0 # Batas clipping pos_weight jika memilih strategi clipped
  log_step_interval: 1 # Interval pencatatan per-step mini-batch (1 = catat setiap step untuk kurva halus)
  num_neighbors: [25, 10] # Sesuai gambar: Ukuran Sampel (k) 25 & 10 (Hop-1 dan Hop-2)

classifier:
  n_estimators: 100
  max_depth: 5
  learning_rate: 0.05

output:
  model_dir: "/kaggle/working/model/exp21/v6"
  result_dir: "/kaggle/working/result/exp21/v6"



In [ ]:
%%writefile config.py
import yaml
from pathlib import Path

def load_config(config_path: str = 'config.reference.yaml') -> dict:
    """
    Memuat file konfigurasi YAML untuk EXP21.
    """
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"Konfigurasi tidak ditemukan di: {path.resolve()}")
        
    with open(path, 'r') as f:
        config = yaml.safe_load(f)
        
    return config



In [ ]:
%%writefile data_preprocessing.py
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import StandardScaler, LabelEncoder
from pathlib import Path

def load_and_preprocess_data(file_path: str, start_year: int = 2016, nrows: int = None) -> tuple:
    """
    Memuat dataset IBM Credit Card Transactions dan memfilter era modern (Year >= start_year).
    Melakukan parsing timestamp kronologis, pembersihan nominal Amount, encoding target,
    dan transformasi kategorikal.
    """
    print(f"Loading data from {file_path} (Filter: Year >= {start_year}, nrows={nrows})...")
    
    cat_columns = ['Use Chip', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?']
    
    # Jika start_year ditentukan dan nrows None/besar, gunakan chunking efisien untuk melewati tahun 1991-2015
    if start_year is not None:
        chunks = []
        collected_rows = 0
        chunksize = 1_000_000
        
        for chunk in pd.read_csv(file_path, chunksize=chunksize):
            # Filter baris dengan tahun >= start_year
            filtered_chunk = chunk[chunk['Year'] >= start_year]
            if len(filtered_chunk) > 0:
                if nrows is not None:
                    remaining = nrows - collected_rows
                    if remaining <= 0:
                        break
                    filtered_chunk = filtered_chunk.iloc[:remaining]
                chunks.append(filtered_chunk)
                collected_rows += len(filtered_chunk)
                if nrows is not None and collected_rows >= nrows:
                    break
                    
        if not chunks:
            raise ValueError(f"Tidak ada baris yang ditemukan dengan Year >= {start_year} pada file {file_path}")
            
        df = pd.concat(chunks, ignore_index=True)
    else:
        df = pd.read_csv(file_path, nrows=nrows)

    print(f"Memproses {len(df)} baris data...")
    print("Parsing timestamps...")
    df['Time'] = df['Time'].fillna('00:00')
    df['datetime'] = pd.to_datetime(
        df['Year'].astype(str) + '-' + 
        df['Month'].astype(str).str.zfill(2) + '-' + 
        df['Day'].astype(str).str.zfill(2) + ' ' + 
        df['Time']
    )
    
    print("Sorting chronologically...")
    df = df.sort_values('datetime').reset_index(drop=True)
    
    print("Cleaning Amount...")
    df['Amount'] = df['Amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
    
    print("Encoding target label...")
    df['Is Fraud?'] = df['Is Fraud?'].map({'Yes': 1, 'No': 0})
    
    print("Encoding categorical features...")
    for col in cat_columns:
        df[col] = df[col].fillna('<missing>').astype(str)
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        
    # Scale Amount menggunakan StandardScaler
    scaler = StandardScaler()
    df['Amount'] = scaler.fit_transform(df[['Amount']])

    # Fitur numerik/kategori untuk node Transaction
    feature_cols = ['Amount'] + cat_columns
    
    return df, feature_cols

def get_temporal_window_split(
    df: pd.DataFrame,
    reference_date: str = '2018-01-01',
    train_days: int = 300,
    val_days: int = 30,
    gap_days: int = 8,
    test_days: int = 60
) -> tuple:
    """
    Membagi dataset berdasarkan protokol rolling window SHINE FRAUD II:
    - Train Window      : [T0, T0 + train_days)
    - Validation Window : [T0 + train_days, T0 + train_days + val_days)
    - Embargo Gap       : [T0 + train_days + val_days, T0 + train_days + val_days + gap_days)
    - Test Window       : [T0 + train_days + val_days + gap_days, T0 + train_days + val_days + gap_days + test_days)
    
    Mengembalikan: (train_df, val_df, test_df, gap_df, stats_dict)
    """
    t0 = pd.to_datetime(reference_date)
    t_val_start = t0 + timedelta(days=train_days)
    t_gap_start = t_val_start + timedelta(days=val_days)
    t_test_start = t_gap_start + timedelta(days=gap_days)
    t_test_end = t_test_start + timedelta(days=test_days)
    
    train_df = df[(df['datetime'] >= t0) & (df['datetime'] < t_val_start)].reset_index(drop=True)
    val_df = df[(df['datetime'] >= t_val_start) & (df['datetime'] < t_gap_start)].reset_index(drop=True)
    gap_df = df[(df['datetime'] >= t_gap_start) & (df['datetime'] < t_test_start)].reset_index(drop=True)
    test_df = df[(df['datetime'] >= t_test_start) & (df['datetime'] < t_test_end)].reset_index(drop=True)
    
    def compute_split_stats(name: str, split_df: pd.DataFrame, start_dt, end_dt):
        n_tx = len(split_df)
        if n_tx == 0:
            return {
                "name": name,
                "start_date": str(start_dt),
                "end_date": str(end_dt),
                "num_transactions": 0,
                "num_fraud": 0,
                "fraud_rate_pct": 0.0,
                "unique_users": 0,
                "unique_merchants": 0,
                "total_nodes": 0
            }
        n_fraud = int(split_df['Is Fraud?'].sum())
        n_users = int(split_df['User'].nunique())
        n_merchants = int(split_df['Merchant Name'].nunique())
        return {
            "name": name,
            "start_date": str(start_dt),
            "end_date": str(end_dt),
            "num_transactions": n_tx,
            "num_fraud": n_fraud,
            "fraud_rate_pct": round((n_fraud / n_tx) * 100, 4),
            "unique_users": n_users,
            "unique_merchants": n_merchants,
            "total_nodes": n_tx + n_users + n_merchants
        }

    total_window_df = pd.concat([train_df, val_df, gap_df, test_df], ignore_index=True)
    
    stats_dict = {
        "protocol": "SHINE FRAUD II Rolling Window Alignment",
        "reference_date_T0": str(t0),
        "durations_days": {
            "train": train_days,
            "validation": val_days,
            "embargo_gap": gap_days,
            "test": test_days,
            "total_horizon": train_days + val_days + gap_days + test_days
        },
        "splits": {
            "train": compute_split_stats("Train", train_df, t0, t_val_start),
            "validation": compute_split_stats("Validation", val_df, t_val_start, t_gap_start),
            "embargo_gap": compute_split_stats("Embargo Gap", gap_df, t_gap_start, t_test_start),
            "test": compute_split_stats("Test", test_df, t_test_start, t_test_end),
            "total_window": compute_split_stats("Total Window", total_window_df, t0, t_test_end)
        }
    }
    
    return train_df, val_df, test_df, gap_df, stats_dict

def generate_rolling_window_indices(
    df: pd.DataFrame,
    start_date: str = '2016-01-01',
    train_days: int = 300,
    val_days: int = 30,
    gap_days: int = 8,
    test_days: int = 60,
    step_days: int = 30
):
    """
    Generator untuk memproduksi beberapa timeframe jendela bergulir (rolling windows).
    """
    total_window_span = timedelta(days=train_days + val_days + gap_days + test_days)
    curr_t0 = pd.to_datetime(start_date)
    max_datetime = df['datetime'].max()
    
    fold_idx = 0
    while curr_t0 + total_window_span <= max_datetime:
        fold_idx += 1
        yield fold_idx, curr_t0.strftime('%Y-%m-%d')
        curr_t0 += timedelta(days=step_days)



In [ ]:
%%writefile graph_builder.py
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData
from sklearn.preprocessing import StandardScaler

def graph_level_undersample(df: pd.DataFrame, ratio: float = 1.0, seed: int = 42) -> pd.DataFrame:
    """
    Melakukan Graph-Level Undersampling jika ratio < 1.0.
    Jika ratio == 1.0 (default di exp21 & SHINE), seluruh data dipertahankan
    tanpa merusak topologi asli jaringan.
    """
    if ratio >= 1.0:
        return df
        
    print(f"Menerapkan graph-level undersampling dengan rasio {ratio}...")
    labels = df['Is Fraud?'].values
    fraud_idx = np.flatnonzero(labels == 1)
    normal_idx = np.flatnonzero(labels == 0)
    
    if len(fraud_idx) == 0 or len(normal_idx) == 0:
        return df
        
    desired_normal = int(np.floor(len(normal_idx) * ratio))
    if desired_normal > len(normal_idx):
        desired_normal = len(normal_idx)
        
    rng = np.random.default_rng(seed)
    chosen_normal = rng.choice(normal_idx, size=desired_normal, replace=False)
    
    selected_idx = np.concatenate([chosen_normal, fraud_idx])
    selected_idx.sort()
    
    return df.iloc[selected_idx].reset_index(drop=True)

def build_hetero_graph(df: pd.DataFrame, feature_cols: list) -> HeteroData:
    """
    Membangun graf heterogen tripartit PyTorch Geometric (HeteroData).
    Nodes:
      - transaction: Fitur transaksi tabular (feature_cols) & label target
      - user: Agregat rata-rata Amount & jumlah transaksi (scaled)
      - merchant: Agregat rata-rata Amount & jumlah transaksi (scaled)
    Edges:
      - (user, buys, transaction) & (transaction, rev_buys, user)
      - (merchant, sells, transaction) & (transaction, rev_sells, merchant)
    """
    print(f"Membangun HeteroData graf ({len(df)} transaksi)...")
    data = HeteroData()
    
    # 1. Pemetaan ID User dan Merchant ke integer kontigu
    user_mapping = {uid: i for i, uid in enumerate(df['User'].unique())}
    merchant_mapping = {mid: i for i, mid in enumerate(df['Merchant Name'].unique())}
    
    num_users = len(user_mapping)
    num_merchants = len(merchant_mapping)
    num_transactions = len(df)
    
    # 2. Agregasi fitur User dan Merchant
    user_agg = df.groupby('User').agg(
        user_avg_amount=('Amount', 'mean'),
        user_tx_count=('Amount', 'count')
    ).reset_index()
    
    merchant_agg = df.groupby('Merchant Name').agg(
        merchant_avg_amount=('Amount', 'mean'),
        merchant_tx_count=('Amount', 'count')
    ).reset_index()
    
    # Pastikan urutan sesuai dengan user_mapping
    user_agg['mapped_id'] = user_agg['User'].map(user_mapping)
    user_agg = user_agg.sort_values('mapped_id')
    
    scaler_user = StandardScaler()
    user_feats_scaled = scaler_user.fit_transform(user_agg[['user_avg_amount', 'user_tx_count']])
    user_features = torch.tensor(user_feats_scaled, dtype=torch.float)
    
    # Pastikan urutan sesuai dengan merchant_mapping
    merchant_agg['mapped_id'] = merchant_agg['Merchant Name'].map(merchant_mapping)
    merchant_agg = merchant_agg.sort_values('mapped_id')
    
    scaler_merchant = StandardScaler()
    merchant_feats_scaled = scaler_merchant.fit_transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
    merchant_features = torch.tensor(merchant_feats_scaled, dtype=torch.float)
    
    data['user'].x = user_features
    data['merchant'].x = merchant_features
    
    # 3. Fitur dan label Transaction
    x_tx = torch.tensor(df[feature_cols].values, dtype=torch.float)
    y_tx = torch.tensor(df['Is Fraud?'].values, dtype=torch.float)
    data['transaction'].x = x_tx
    data['transaction'].y = y_tx
    data['transaction'].time = torch.arange(num_transactions)
    
    # 4. Konstruksi Edges (Bipartit / Tripartit dua arah)
    user_src = [user_mapping[uid] for uid in df['User']]
    tx_dst = list(range(num_transactions))
    edge_index_user_tx = torch.tensor([user_src, tx_dst], dtype=torch.long)
    
    merchant_src = [merchant_mapping[mid] for mid in df['Merchant Name']]
    edge_index_merchant_tx = torch.tensor([merchant_src, tx_dst], dtype=torch.long)
    
    data['user', 'buys', 'transaction'].edge_index = edge_index_user_tx
    data['merchant', 'sells', 'transaction'].edge_index = edge_index_merchant_tx
    
    # Reverse edges untuk propagasi pesan dua arah pada GraphSAGE
    data['transaction', 'rev_buys', 'user'].edge_index = edge_index_user_tx.flip([0])
    data['transaction', 'rev_sells', 'merchant'].edge_index = edge_index_merchant_tx.flip([0])
    
    print(f"Graf selesai dibangun: {num_users} users, {num_merchants} merchants, {num_transactions} transaksi.")
    return data



In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv

class HeteroGraphSAGE(nn.Module):
    def __init__(
        self, 
        hidden_channels: int, 
        out_channels: int, 
        num_layers: int, 
        data_metadata: tuple, 
        feature_dims: dict, 
        aggr: str = 'mean',
        dropout: float = 0.2
    ):
        super().__init__()
        
        self.num_layers = num_layers
        self.aggr = aggr
        self.dropout_rate = dropout
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        
        # Inisialisasi proyeksi linear untuk tiap tipe node
        self.lin_embeddings = nn.ModuleDict()
        self.param_embeddings = nn.ParameterDict()
        for node_type, dim in feature_dims.items():
            if dim == 0:
                self.param_embeddings[node_type] = nn.Parameter(torch.randn(1, hidden_channels))
            else:
                self.lin_embeddings[node_type] = nn.Linear(dim, hidden_channels)
                
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv(
                    in_channels=(-1, -1), # Lazy initialization
                    out_channels=hidden_channels,
                    aggr=aggr
                )
                for edge_type in data_metadata[1]
            })
            self.convs.append(conv)
            
            # BatchNorm untuk node transaction
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        # Classifier linear
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, num_nodes_dict, return_embeddings=False):
        # 1. Proyeksi fitur awal
        h_dict = {}
        
        # Node yang memiliki fitur (transaction, user, merchant)
        for node_type, x in x_dict.items():
            if node_type in self.lin_embeddings:
                h_dict[node_type] = self.lin_embeddings[node_type](x)
                
        # Node tanpa fitur jika ada
        for node_type, param in self.param_embeddings.items():
            num_nodes = num_nodes_dict[node_type]
            h_dict[node_type] = param.expand(num_nodes, -1)
                
        # 2. Message Passing
        for i in range(self.num_layers):
            h_dict = self.convs[i](h_dict, edge_index_dict)
            
            # ReLU, BatchNorm, dan Dropout pada transaction nodes
            for node_type in h_dict.keys():
                h_dict[node_type] = F.relu(h_dict[node_type])
                if node_type == 'transaction':
                    h_dict[node_type] = self.bns[i](h_dict[node_type])
                    h_dict[node_type] = self.dropout(h_dict[node_type])
        
        if return_embeddings:
            return h_dict['transaction']
            
        # 3. Klasifikasi biner untuk transaction nodes
        out = self.classifier(h_dict['transaction'])
        return torch.sigmoid(out).squeeze(-1)

def manual_weighted_bce(probs, targets, pos_weight, label_smoothing: float = 0.0):
    """
    Class-weighted Binary Cross-Entropy Loss untuk output probabilitas sigmoid.
    pos_weight = (num_negatives / (num_positives + 1e-7)) atau versi yang diredam (sqrt/clipped).
    Catatan: label_smoothing dinonaktifkan (0.0) pada v5 untuk mencegah fatal mathematical
    collision dengan pos_weight yang memicu mode collapse & loss stagnation.
    """
    if label_smoothing > 0.0:
        targets_smooth = targets * (1.0 - label_smoothing) + 0.5 * label_smoothing
    else:
        targets_smooth = targets
        
    loss = - (pos_weight * targets_smooth * torch.log(probs + 1e-7) + 
              (1.0 - targets_smooth) * torch.log(1.0 - probs + 1e-7))
    return loss.mean()

def manual_unweighted_bce(probs, targets):
    """
    Standar Binary Cross-Entropy Loss tanpa pembobotan kelas (untuk monitoring kurva murni).
    """
    loss = - (targets * torch.log(probs + 1e-7) + 
              (1.0 - targets) * torch.log(1.0 - probs + 1e-7))
    return loss.mean()

def focal_loss(probs, targets, alpha: float = 0.25, gamma: float = 2.0):
    """
    Focal Loss untuk menangani class imbalance ekstrim tanpa memicu exploding loss.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    """
    p = probs + 1e-7
    p_t = targets * p + (1.0 - targets) * (1.0 - p)
    alpha_t = targets * alpha + (1.0 - targets) * (1.0 - alpha)
    loss = - alpha_t * ((1.0 - p_t) ** gamma) * torch.log(p_t)
    return loss.mean()



In [ ]:
%%writefile samplers.py
import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

def calculate_degree_centrality(data: HeteroData, edge_type: tuple):
    """Menghitung degree centrality sederhana untuk edge type tertentu."""
    src, rel, dst = edge_type
    edge_index = data[edge_type].edge_index
    num_nodes = data[src].num_nodes
    degree = torch.zeros(num_nodes, dtype=torch.float)
    degree.scatter_add_(0, edge_index[0], torch.ones(edge_index.shape[1]))
    return degree

def apply_uniform_sampling(data: HeteroData):
    """
    Baseline: Uniform Random Sampling.
    Memberikan probabilitas yang sama (1.0) untuk semua edge.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        num_edges = data[edge_type].edge_index.size(1)
        data[edge_type].edge_weight = torch.ones(num_edges, dtype=torch.float)
    return data

def apply_topology_aware_sampling(data: HeteroData):
    """
    Topology-Aware Sampling (Inverse Log-Degree Centrality - v4 Stabilized).
    Menggunakan inverse sublinear log1p(degree) untuk memprioritaskan edge struktural
    tanpa disparitas bobot yang ekstrem.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        # Sublinear scaling: log1p(degree)
        log_deg = torch.log1p(deg_src[src_nodes])
        weights = 1.0 / (log_deg + 1e-5)
        min_val = weights.min()
        max_val = weights.max()
        if (max_val - min_val) > 1e-5:
            weights = (weights - min_val) / (max_val - min_val)
        else:
            weights = torch.zeros_like(weights)
        data[edge_type].edge_weight = weights + 0.1
    return data

def apply_importance_based_sampling(data: HeteroData):
    """
    Importance-Based Sampling (Log-Degree Centrality Proxy - v4 Stabilized).
    Menggunakan transformasi sublinear log1p untuk meredam varians sampling
    akibat super-hub merchant dan mencegah anomali ledakan False Positive di akhir training.
    """
    first_edge = data.edge_types[0]
    if hasattr(data[first_edge], 'edge_weight') and data[first_edge].edge_weight is not None:
        return data
    for edge_type in data.edge_types:
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        # Sublinear scaling: log1p(degree) meredam lonjakan super-hubs
        log_deg = torch.log1p(deg_src[src_nodes])
        min_val = log_deg.min()
        max_val = log_deg.max()
        if (max_val - min_val) > 1e-5:
            importance_weights = (log_deg - min_val) / (max_val - min_val)
        else:
            importance_weights = torch.zeros_like(log_deg)
        data[edge_type].edge_weight = importance_weights + 0.1
    return data

def get_neighbor_loader(data: HeteroData, batch_size: int = 1024, num_neighbors: list = [15, 10], method: str = 'uniform', shuffle: bool = True, input_nodes=None, num_workers: int = 0):
    """
    Mengembalikan PyG NeighborLoader berdasarkan strategi neighbor sampling yang dipilih.
    """
    if method == 'uniform':
        data = apply_uniform_sampling(data)
    elif method == 'topology':
        data = apply_topology_aware_sampling(data)
    elif method == 'importance':
        data = apply_importance_based_sampling(data)
    else:
        raise ValueError(f"Metode sampling tidak dikenal: {method}")

    if input_nodes is None:
        input_nodes = ('transaction', torch.arange(data['transaction'].num_nodes))

    loader = NeighborLoader(
        data,
        num_neighbors=num_neighbors,
        batch_size=batch_size,
        input_nodes=input_nodes,
        shuffle=shuffle,
        weight_attr='edge_weight',
        num_workers=num_workers
    )
    return loader



In [ ]:
%%writefile visualize_results.py
import argparse
import json
import os
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Headless rendering for server/CLI
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from sklearn.metrics import precision_recall_curve, roc_curve

# Set font & style standard
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.autolayout'] = False

# Palette warna harmonis dan profesional (Academic / Modern Analytics)
COLOR_UNIFORM = "#3B82F6"      # Royal Blue
COLOR_TOPOLOGY = "#10B981"     # Emerald Green
COLOR_IMPORTANCE = "#F59E0B"   # Amber Orange
METHOD_COLORS = {
    'uniform': COLOR_UNIFORM,
    'topology': COLOR_TOPOLOGY,
    'importance': COLOR_IMPORTANCE
}
METHOD_NAMES = {
    'uniform': 'Uniform (Baseline)',
    'topology': 'Topology-Aware',
    'importance': 'Importance-Based'
}

def load_data(result_dir: Path):
    """Memuat seluruh data hasil eksperimen dan statistik window."""
    methods = ['uniform', 'topology', 'importance']
    results = {}
    for m in methods:
        p = result_dir / f"exp21_{m}_results.json"
        if not p.exists():
            raise FileNotFoundError(f"File hasil tidak ditemukan: {p}")
        with open(p, 'r') as f:
            results[m] = json.load(f)
            
    stat_path = result_dir / "window_statistics.json"
    window_stats = None
    if stat_path.exists():
        with open(stat_path, 'r') as f:
            window_stats = json.load(f)
            
    return results, window_stats

def plot_chart1_primary_metrics(results, out_dir: Path):
    """
    Chart 1: Grouped Bar Chart - Perbandingan Metrik Utama (F1, Recall, Precision, AUCPR, ROC-AUC)
    """
    fig, ax = plt.subplots(figsize=(10.5, 6), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    metrics = ['f1_score', 'recall', 'precision', 'auprc', 'roc_auc']
    metric_labels = ['F1-Score', 'Recall', 'Precision', 'AUCPR (AUPRC)', 'ROC-AUC']
    methods = ['uniform', 'topology', 'importance']
    
    x = np.arange(len(metric_labels))
    width = 0.26
    
    for i, m in enumerate(methods):
        opt = results[m]['metrics_optimal_threshold']
        ind = results[m]['metrics_threshold_independent']
        vals = [
            opt['f1_score'],
            opt['recall'],
            opt['precision'],
            ind['auprc'],
            ind['roc_auc']
        ]
        offset = (i - 1) * width
        rects = ax.bar(x + offset, vals, width, label=METHOD_NAMES[m], 
                       color=METHOD_COLORS[m], edgecolor='#1E293B', linewidth=0.8, alpha=0.92)
        
        # Tambahkan nilai di atas bar
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.4f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 4),
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=8.5, fontweight='bold',
                        color='#1E293B', rotation=0)

    ax.set_title("EXP21: Komparasi Metrik Klasifikasi Utama antar Strategi Sampling\n"
                 "(Evaluasi Test Set 60 Hari, 283.529 Transaksi, 421 Fraud)",
                 fontsize=13, fontweight='bold', pad=15, color='#0F172A')
    ax.set_ylabel("Nilai Metrik (0.0 - 1.0)", fontsize=11, fontweight='semibold', labelpad=8)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, fontsize=10.5, fontweight='semibold')
    ax.set_ylim(0.80, 1.03)  # Zoom in rentang metrik tinggi agar kontras terlihat jelas
    ax.grid(axis='y', linestyle='--', alpha=0.5, color='#CBD5E1')
    ax.legend(loc='lower right', framealpha=0.95, facecolor='#FFFFFF', edgecolor='#CBD5E1', fontsize=10)
    
    # Anotasi keunggulan
    ax.annotate('Topology: Recall Sempurna (100.0%)\n& F1 Tertinggi (0.9397)', 
                xy=(1, 1.0), xytext=(0.85, 0.83),
                arrowprops=dict(facecolor='#10B981', shrink=0.08, width=1.5, headwidth=6),
                bbox=dict(boxstyle="round,pad=0.4", fc="#ECFDF5", ec="#10B981", lw=1.2),
                fontsize=9, fontweight='semibold', color='#065F46')

    plt.tight_layout()
    out_path = out_dir / "chart1_primary_metrics_comparison.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart2_threshold_calibration(results, out_dir: Path):
    """
    Chart 2: Paired Bar Chart - Default Threshold (0.5) vs Calibrated Optimal Threshold (tau*)
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 5.2), dpi=300, sharey=True)
    fig.patch.set_facecolor('#FFFFFF')
    
    methods = ['uniform', 'topology', 'importance']
    metrics = ['f1_score', 'recall', 'precision']
    labels = ['F1-Score', 'Recall', 'Precision']
    x = np.arange(len(labels))
    width = 0.35
    
    for idx, m in enumerate(methods):
        ax = axes[idx]
        ax.set_facecolor('#F8FAFC')
        
        m_def = results[m]['metrics_default_threshold_05']
        m_opt = results[m]['metrics_optimal_threshold']
        tau_val = results[m]['threshold_calibration']['optimal_tau']
        
        vals_def = [m_def['f1_score'], m_def['recall'], m_def['precision']]
        vals_opt = [m_opt['f1_score'], m_opt['recall'], m_opt['precision']]
        
        rects1 = ax.bar(x - width/2, vals_def, width, label='Default (τ = 0.50)', 
                        color='#94A3B8', edgecolor='#334155', linewidth=0.7)
        rects2 = ax.bar(x + width/2, vals_opt, width, label=f'Optimal (τ* = {tau_val:.3f})', 
                        color=METHOD_COLORS[m], edgecolor='#0F172A', linewidth=0.8)
        
        for rect in rects1:
            h = rect.get_height()
            ax.annotate(f'{h:.3f}', xy=(rect.get_x() + rect.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points", ha='center', va='bottom',
                        fontsize=8, color='#475569')
            
        for rect in rects2:
            h = rect.get_height()
            ax.annotate(f'{h:.3f}', xy=(rect.get_x() + rect.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points", ha='center', va='bottom',
                        fontsize=8, fontweight='bold', color='#0F172A')
            
        ax.set_title(f"{METHOD_NAMES[m]}\nOptimal Threshold τ* = {tau_val:.3f}", 
                     fontsize=11.5, fontweight='bold', pad=10, color='#0F172A')
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=10, fontweight='semibold')
        ax.set_ylim(0.82, 1.04)
        ax.grid(axis='y', linestyle='--', alpha=0.45, color='#CBD5E1')
        ax.legend(loc='lower left', fontsize=9, framealpha=0.9)
        
    axes[0].set_ylabel("Nilai Metrik (0.0 - 1.0)", fontsize=11, fontweight='semibold')
    plt.suptitle("EXP21: Pengaruh Kalibrasi Threshold Optimal (τ*) vs Default (τ = 0.50)", 
                 fontsize=13.5, fontweight='bold', y=1.03, color='#0F172A')
    
    plt.tight_layout()
    out_path = out_dir / "chart2_threshold_calibration_effect.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart3_confusion_matrix(results, window_stats, out_dir: Path):
    """
    Chart 3: Multi-Subplot Confusion Matrix Heatmaps (Uniform vs Topology vs Importance)
    Dihitung eksak secara analitik dari data metrik.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.8), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    
    test_total = window_stats['splits']['test']['num_transactions']
    test_fraud = window_stats['splits']['test']['num_fraud']
    test_normal = test_total - test_fraud
    
    methods = ['uniform', 'topology', 'importance']
    cm_data = {}
    
    for m in methods:
        opt = results[m]['metrics_optimal_threshold']
        rec = opt['recall']
        prec = opt['precision']
        
        tp = int(round(test_fraud * rec))
        fn = test_fraud - tp
        fp = int(round(tp / prec - tp))
        tn = test_normal - fp
        cm_data[m] = np.array([[tn, fp], [fn, tp]])
        
    for idx, m in enumerate(methods):
        ax = axes[idx]
        cm = cm_data[m]
        tn, fp, fn, tp = cm.ravel()
        
        # Color matrix berdasarkan sampling
        cmap = plt.cm.Blues if m == 'uniform' else (plt.cm.Greens if m == 'topology' else plt.cm.Oranges)
        # Normalisasi log-scale untuk visual kontras mengingat disparitas kelas 283.000 vs 400
        cm_display = np.log1p(cm)
        im = ax.imshow(cm_display, interpolation='nearest', cmap=cmap, alpha=0.85)
        
        labels_text = [
            [f"TN\n{tn:,}\n({tn/test_normal*100:.2f}%)", f"FP\n{fp:,}\n({fp/test_normal*100:.2f}%)"],
            [f"FN (Missed)\n{fn:,}\n({fn/test_fraud*100:.2f}%)", f"TP (Caught)\n{tp:,}\n({tp/test_fraud*100:.2f}%)"]
        ]
        
        for i in range(2):
            for j in range(2):
                text_color = '#FFFFFF' if (i == 0 and j == 0) or (i == 1 and j == 1) else '#0F172A'
                # Highlight FN = 0 dengan warna hijau jika 0
                if i == 1 and j == 0 and fn == 0:
                    text_color = '#047857'
                    font_w = 'bold'
                else:
                    font_w = 'bold' if (i == 1 and j == 1) or (i == 0 and j == 0) else 'semibold'
                    
                ax.text(j, i, labels_text[i][j], ha="center", va="center",
                        color=text_color, fontsize=10, fontweight=font_w)

        ax.set_title(f"{METHOD_NAMES[m]}\n(Total Fraud = {test_fraud}, Normal = {test_normal:,})",
                     fontsize=11.5, fontweight='bold', pad=10, color='#0F172A')
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(['Pred: Normal', 'Pred: Fraud'], fontsize=9.5, fontweight='semibold')
        ax.set_yticklabels(['Aktual: Normal', 'Aktual: Fraud'], fontsize=9.5, fontweight='semibold')
        
    plt.suptitle("EXP21: Heatmap Confusion Matrix pada Test Set (283.529 Transaksi)\n"
                 "Topology-Aware Berhasil Mencapai FN = 0 (Zero Missed Fraud)",
                 fontsize=13.5, fontweight='bold', y=1.04, color='#0F172A')
    plt.tight_layout()
    out_path = out_dir / "chart3_confusion_matrix_comparison.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart4_banking_operations(results, window_stats, out_dir: Path):
    """
    Chart 4: Evaluasi Operasional Perbankan - Alert Triage TP@500 & Precision@500 vs Total Real Fraud
    """
    fig, ax = plt.subplots(figsize=(9, 5.5), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    methods = ['uniform', 'topology', 'importance']
    names = [METHOD_NAMES[m] for m in methods]
    tp_vals = [results[m]['metrics_threshold_independent']['tp_at_500'] for m in methods]
    prec_vals = [results[m]['metrics_threshold_independent']['precision_at_500'] * 100 for m in methods]
    total_fraud = window_stats['splits']['test']['num_fraud']
    
    x = np.arange(len(methods))
    bars = ax.bar(x, tp_vals, width=0.48, color=[METHOD_COLORS[m] for m in methods], 
                  edgecolor='#1E293B', linewidth=1.0, alpha=0.92)
    
    # Garis acuan total fraud di test set
    ax.axhline(total_fraud, color='#DC2626', linestyle='--', linewidth=1.8, 
               label=f'Total Fraud Nyata di Test Set ({total_fraud} Kasus)')
    
    for i, bar in enumerate(bars):
        h = bar.get_height()
        pct = prec_vals[i]
        caught_pct = (h / total_fraud) * 100
        ax.annotate(f"{h} / 500\nPrecision: {pct:.1f}%\n({caught_pct:.1f}% Total Fraud)",
                    xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 6), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#0F172A')
        
    ax.set_title("EXP21: Efektivitas Alert Triage Harian Perbankan (TP@500 & Precision@500)\n"
                 "Hasil Evaluasi pada Top-500 Transaksi Paling Berisiko",
                 fontsize=12.5, fontweight='bold', pad=15, color='#0F172A')
    ax.set_ylabel("Jumlah True Positive Tertangkap (Kasus)", fontsize=11, fontweight='semibold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=10.5, fontweight='semibold')
    ax.set_ylim(350, 475)
    ax.grid(axis='y', linestyle='--', alpha=0.5, color='#CBD5E1')
    ax.legend(loc='lower right', fontsize=10, framealpha=0.95, facecolor='#FFFFFF', edgecolor='#CBD5E1')
    
    plt.tight_layout()
    out_path = out_dir / "chart4_banking_triage_tp500.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart5_lift_percentile(results, out_dir: Path):
    """
    Chart 5: Lift Factor (Lift@1%) - Efektivitas Peningkatan Deteksi Dibanding Random Inspection
    """
    fig, ax = plt.subplots(figsize=(8.5, 5.2), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    methods = ['uniform', 'topology', 'importance']
    names = [METHOD_NAMES[m] for m in methods]
    lift_vals = [results[m]['metrics_threshold_independent']['lift_1_percent'] for m in methods]
    
    bars = ax.bar(names, lift_vals, width=0.45, color=[METHOD_COLORS[m] for m in methods],
                  edgecolor='#1E293B', linewidth=1.0, alpha=0.92)
    
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.2f}x",
                    xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 5), textcoords="offset points",
                    ha='center', va='bottom', fontsize=11, fontweight='bold', color='#0F172A')
        
    # Baseline random inspection = 1.0x
    ax.axhline(1.0, color='#64748B', linestyle=':', linewidth=1.5, label='Baseline Random Review (1.0x)')
    
    ax.set_title("EXP21: Peningkatan Deteksi Fraud pada Top-1% Transaksi Teratas (Lift@1%)\n"
                 "Daya Ungkit Sistem Dibandingkan Audit Transaksi Secara Acak",
                 fontsize=12.5, fontweight='bold', pad=15, color='#0F172A')
    ax.set_ylabel("Faktor Pengali Lift (x)", fontsize=11, fontweight='semibold')
    ax.set_ylim(80, 112)
    ax.grid(axis='y', linestyle='--', alpha=0.5, color='#CBD5E1')
    ax.legend(loc='lower right', fontsize=10, framealpha=0.95, facecolor='#FFFFFF')
    
    plt.tight_layout()
    out_path = out_dir / "chart5_lift_at_1_percent.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart6_runtime_breakdown(results, out_dir: Path):
    """
    Chart 6: Stacked Horizontal Bar Chart - Computation Runtime Breakdown
    """
    fig, ax = plt.subplots(figsize=(11, 5.8), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    methods = ['uniform', 'topology', 'importance']
    names = [METHOD_NAMES[m] for m in methods]
    
    prep_times = [results[m]['runtime_seconds']['data_preprocessing'] for m in methods]
    gnn_times = [results[m]['runtime_seconds']['gnn_training'] for m in methods]
    xgb_times = [results[m]['runtime_seconds']['xgb_training'] for m in methods]
    inf_times = [results[m]['runtime_seconds']['total_test_inference'] for m in methods]
    total_times = [results[m]['runtime_seconds']['total_pipeline'] for m in methods]
    
    y = np.arange(len(methods))
    height = 0.42
    
    p1 = ax.barh(y, prep_times, height, label='Data Prep (Graph Construction)', color='#64748B', edgecolor='#1E293B')
    p2 = ax.barh(y, gnn_times, height, left=prep_times, label='GraphSAGE Training (10 Epochs)', color='#3B82F6', edgecolor='#1E293B')
    left_xgb = np.array(prep_times) + np.array(gnn_times)
    p3 = ax.barh(y, xgb_times, height, left=left_xgb, label='XGBoost Training (100 Trees)', color='#10B981', edgecolor='#1E293B')
    left_inf = left_xgb + np.array(xgb_times)
    p4 = ax.barh(y, inf_times, height, left=left_inf, label='Test Inference (GNN + XGB)', color='#F59E0B', edgecolor='#1E293B')
    
    for i, tot in enumerate(total_times):
        ax.annotate(f"Total: {tot:.1f}s ({tot/60:.1f}m)",
                    xy=(tot, y[i]), xytext=(12, 0), textcoords="offset points",
                    ha='left', va='center', fontsize=10, fontweight='bold', color='#0F172A')
        
    ax.set_title("EXP21: Analisis Efisiensi Waktu Eksekusi Pipeline (Kaggle GPU P100)\n"
                 "Bukti Sukses Eliminasi Timeout: Seluruh Metode Selesai dalam ~14 - 15 Menit",
                 fontsize=12.5, fontweight='bold', pad=15, color='#0F172A')
    ax.set_xlabel("Waktu Komputasi (Detik)", fontsize=11, fontweight='semibold')
    ax.set_yticks(y)
    ax.set_yticklabels(names, fontsize=10.5, fontweight='semibold')
    ax.set_xlim(0, 1200)
    ax.grid(axis='x', linestyle='--', alpha=0.5, color='#CBD5E1')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=4, fontsize=9.2, 
              framealpha=0.95, facecolor='#FFFFFF', edgecolor='#CBD5E1')
    
    plt.tight_layout()
    out_path = out_dir / "chart6_runtime_breakdown.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart7_temporal_window_timeline(window_stats, out_dir: Path):
    """
    Chart 7: Gantt-style Timeline Chart - Desain Temporal Rolling Window (SHINE FRAUD II Protocol)
    """
    fig, ax = plt.subplots(figsize=(12, 5.0), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    splits = [
        ('Train Set (300 Hari)', '2018-01-01', '2018-10-28', '#3B82F6', '1.415.044 Transaksi | 1.930 Fraud (0.136%)'),
        ('Validation Set (30 Hari)', '2018-10-28', '2018-11-27', '#10B981', '141.270 Transaksi | 291 Fraud (0.206%)\nTuning Threshold tau*'),
        ('Embargo Gap (8 Hari)', '2018-11-27', '2018-12-05', '#EF4444', '37.515 Tx (Isolasi Label Leakage)'),
        ('Test Set (60 Hari)', '2018-12-05', '2019-02-03', '#8B5CF6', '283.529 Transaksi | 421 Fraud (0.149%)\nEvaluasi Induktif Murni')
    ]
    
    y_positions = [3, 2, 1, 0]
    
    for idx, (name, s_date, e_date, col, desc) in enumerate(splits):
        y = y_positions[idx]
        d_start = datetime.strptime(s_date, '%Y-%m-%d')
        d_end = datetime.strptime(e_date, '%Y-%m-%d')
        dur = (d_end - d_start).days
        
        ax.barh(y, dur, left=mdates.date2num(d_start), height=0.45, color=col, 
                edgecolor='#0F172A', linewidth=1.0, alpha=0.9)
        
        # Anotasi teks di dalam/samping bar
        mid_date = d_start + timedelta(days=dur/2)
        if dur > 15:
            ax.text(mdates.date2num(mid_date), y, f"{dur} Hari", 
                    ha='center', va='center', color='#FFFFFF', fontweight='bold', fontsize=9.5)
        else:
            # Letakkan di atas bar jika bar terlalu sempit
            ax.text(mdates.date2num(mid_date), y + 0.32, f"{dur} Hari", 
                    ha='center', va='bottom', color='#DC2626', fontweight='bold', fontsize=9.0)
        
        # Keterangan detail
        if idx == 3:
            # Test set
            ax.text(mdates.date2num(d_start) - 6, y, f"{name}\n{desc}", ha='right', va='center', 
                    color='#1E293B', fontsize=8.5, fontweight='semibold')
        else:
            ax.text(mdates.date2num(d_end) + 6, y, f"{name}\n{desc}", ha='left', va='center', 
                    color='#1E293B', fontsize=8.5, fontweight='semibold')
        
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    
    ax.set_yticks(y_positions)
    ax.set_yticklabels(['Train Set', 'Validation Set', 'Embargo Gap', 'Test Set'], fontsize=10, fontweight='semibold')
    ax.set_title("EXP21: Desain Temporal Rolling Window Alignment (SHINE FRAUD II Protocol)\n"
                 "Mitigasi Concept Drift (2018-2019) & Pencegahan Data Leakage dengan Embargo Gap",
                 fontsize=12.5, fontweight='bold', pad=15, color='#0F172A')
    ax.set_xlabel("Lini Masa Waktu (Tanggal Transaksi)", fontsize=10.5, fontweight='semibold', labelpad=8)
    ax.grid(axis='x', linestyle='--', alpha=0.5, color='#CBD5E1')
    ax.set_xlim(mdates.date2num(datetime(2017, 11, 15)), mdates.date2num(datetime(2019, 3, 20)))
    
    plt.tight_layout()
    out_path = out_dir / "chart7_temporal_window_timeline.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart8_data_distribution(window_stats, out_dir: Path):
    """
    Chart 8: Dual-Axis Chart - Volume Transaksi & Proporsi Fraud Rate per Split Temporal
    """
    fig, ax1 = plt.subplots(figsize=(10, 5.2), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax1.set_facecolor('#F8FAFC')
    
    splits_keys = ['train', 'validation', 'embargo_gap', 'test']
    labels = ['Train (300d)', 'Validation (30d)', 'Embargo Gap (8d)', 'Test (60d)']
    
    volumes = [window_stats['splits'][k]['num_transactions'] for k in splits_keys]
    fraud_counts = [window_stats['splits'][k]['num_fraud'] for k in splits_keys]
    fraud_rates = [window_stats['splits'][k]['fraud_rate_pct'] for k in splits_keys]
    
    x = np.arange(len(labels))
    width = 0.42
    
    bars = ax1.bar(x, volumes, width, color='#3B82F6', edgecolor='#1E293B', alpha=0.88, label='Volume Transaksi')
    ax1.set_ylabel("Volume Transaksi (Log Scale)", fontsize=11, fontweight='semibold', color='#1E40AF')
    ax1.set_yscale('log')
    ax1.set_ylim(1e4, 3e6)
    ax1.tick_params(axis='y', labelcolor='#1E40AF')
    
    for i, bar in enumerate(bars):
        h = bar.get_height()
        ax1.annotate(f"{h:,}\n({fraud_counts[i]:,} Fraud)",
                     xy=(bar.get_x() + bar.get_width() / 2, h),
                     xytext=(0, 4), textcoords="offset points",
                     ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='#1E3A8A')
        
    ax2 = ax1.twinx()
    line = ax2.plot(x, fraud_rates, color='#DC2626', marker='o', linewidth=2.4, markersize=8, label='Fraud Rate (%)')
    ax2.set_ylabel("Fraud Rate (%)", fontsize=11, fontweight='semibold', color='#DC2626')
    ax2.tick_params(axis='y', labelcolor='#DC2626')
    ax2.set_ylim(0.08, 0.28)
    
    for i, rate in enumerate(fraud_rates):
        ax2.annotate(f"{rate:.3f}%", xy=(x[i], rate), xytext=(0, 8), textcoords="offset points",
                     ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#B91C1C')
        
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, fontsize=10.5, fontweight='semibold')
    ax1.set_title("EXP21: Karakteristik Volume Transaksi dan Rasio Fraud Tiap Partisi Temporal\n"
                 "(Konsistensi Rasio Fraud ~0.13% - 0.21% Menjamin Eliminasi Concept Drift)",
                 fontsize=12.5, fontweight='bold', pad=15, color='#0F172A')
    
    # Gabungkan legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', framealpha=0.95, facecolor='#FFFFFF')
    
    plt.tight_layout()
    out_path = out_dir / "chart8_data_split_characteristics.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart9_radar_synthesis(results, out_dir: Path):
    """
    Chart 9: Spider / Radar Chart - Evaluasi Multi-Kriteria Menyeluruh (7 Dimensi)
    """
    categories = ['F1-Score', 'Recall', 'Precision', 'AUCPR', 'ROC-AUC', 'Precision@500', 'Efisiensi Waktu']
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(9.2, 9.2), subplot_kw=dict(polar=True), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')
    
    methods = ['uniform', 'topology', 'importance']
    
    # Hitung normalisasi skor untuk tiap dimensi
    max_runtime = 950.0  # Waktu maksimum untuk pembalik kecepatan
    
    for m in methods:
        opt = results[m]['metrics_optimal_threshold']
        ind = results[m]['metrics_threshold_independent']
        rt = results[m]['runtime_seconds']['total_pipeline']
        
        # Skor efisiensi waktu dinormalisasi (semakin cepat mendekati 1.0)
        speed_score = (max_runtime - rt) / 150.0 + 0.8  # Skala ~0.9 - 1.0
        
        values = [
            opt['f1_score'],
            opt['recall'],
            opt['precision'],
            ind['auprc'],
            ind['roc_auc'],
            ind['precision_at_500'],
            speed_score
        ]
        values += values[:1]
        
        ax.plot(angles, values, linewidth=2.4, linestyle='solid', label=METHOD_NAMES[m], color=METHOD_COLORS[m])
        ax.fill(angles, values, color=METHOD_COLORS[m], alpha=0.18)
        
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=10.5, fontweight='bold', color='#1E293B')
    ax.set_ylim(0.80, 1.02)
    ax.set_yticks([0.85, 0.90, 0.95, 1.00])
    ax.set_yticklabels(['0.85', '0.90', '0.95', '1.00'], fontsize=8.5, color='#64748B')
    ax.grid(color='#CBD5E1', linestyle='--', linewidth=0.8)
    
    ax.set_title("EXP21: Radar Sintesis Evaluasi Multi-Dimensi Antar Strategi Sampling\n"
                 "(Performa Deteksi, Presisi, Robustness, dan Efisiensi Komputasi)",
                 fontsize=12.5, fontweight='bold', pad=35, color='#0F172A')
    ax.legend(loc='upper right', bbox_to_anchor=(1.20, 1.05), fontsize=10, 
              framealpha=0.95, facecolor='#FFFFFF', edgecolor='#CBD5E1')
    
    plt.tight_layout()
    out_path = out_dir / "chart9_radar_multicriteria_synthesis.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def load_predictions(result_dir: Path):
    """Memuat file prediksi mentah (.npz) untuk ketiga metode jika tersedia."""
    methods = ['uniform', 'topology', 'importance']
    predictions = {}
    for m in methods:
        p = result_dir / f"exp21_{m}_predictions.npz"
        if p.exists():
            data = np.load(p)
            predictions[m] = {
                'probs': data['test_probs'],
                'labels': data['test_labels']
            }
        else:
            return None
    return predictions

def load_histories(result_dir: Path):
    """Memuat file riwayat training per-epoch & per-iteration (.json) jika tersedia."""
    methods = ['uniform', 'topology', 'importance']
    histories = {}
    for m in methods:
        p = result_dir / f"exp21_{m}_history.json"
        if p.exists():
            with open(p, 'r') as f:
                histories[m] = json.load(f)
        else:
            return None
    return histories

def plot_chart10_precision_recall_curves(results, predictions, window_stats, out_dir: Path):
    """
    Chart 10: Kurva Kontinu Precision-Recall (PR Curve) Berlekuk
    Menampilkan trade-off presisi dan sensitivitas pada sebaran data imbalanced ekstrim,
    dilengkapi titik operasi threshold optimal tau* dan garis baseline no-skill.
    """
    fig, ax = plt.subplots(figsize=(10, 7.2), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')

    methods = ['uniform', 'topology', 'importance']
    
    # Baseline No-Skill line (Rasio fraud asli pada Test Set)
    test_fraud_rate = 0.001538
    if window_stats and 'splits' in window_stats and 'test' in window_stats['splits']:
        test_fraud_rate = window_stats['splits']['test']['fraud_rate_pct'] / 100.0
    
    ax.axhline(y=test_fraud_rate, color='#94A3B8', linestyle='--', linewidth=1.5,
               label=f'Baseline No-Skill (Prevalensi Fraud: {test_fraud_rate*100:.3f}%)')

    for m in methods:
        probs = predictions[m]['probs']
        labels = predictions[m]['labels']
        tau_star = results[m]['metrics_optimal_threshold']['threshold']
        auprc_val = results[m]['metrics_threshold_independent']['auprc']
        
        precisions, recalls, thresholds = precision_recall_curve(labels, probs)
        
        # Plot kurva PR kontinu
        color = METHOD_COLORS[m]
        name = METHOD_NAMES[m]
        ax.plot(recalls, precisions, color=color, linewidth=2.6, 
                label=f'{name} (AUPRC = {auprc_val:.4f})')
        
        # Cari titik operasi pada tau*
        preds_tau = (probs >= tau_star).astype(int)
        tp = np.sum((preds_tau == 1) & (labels == 1))
        fp = np.sum((preds_tau == 1) & (labels == 0))
        fn = np.sum((preds_tau == 0) & (labels == 1))
        pt_rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        pt_prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        # Tandai titik operasi tau*
        ax.plot(pt_rec, pt_prec, marker='*', markersize=14, color=color, 
                markeredgecolor='#0F172A', markeredgewidth=1.2, zorder=5)
        
        # Offset anotasi agar tidak tumpang tindih
        y_offset = 12 if m == 'uniform' else (-20 if m == 'topology' else -35)
        x_offset = -15 if m == 'topology' else 10
        ax.annotate(
            f"{m.capitalize()} $\\tau^*={tau_star:.3f}$\n(Rec: {pt_rec*100:.1f}%, Prec: {pt_prec*100:.1f}%)",
            xy=(pt_rec, pt_prec), xytext=(x_offset, y_offset), textcoords='offset points',
            fontsize=8.5, fontweight='bold', color=color,
            arrowprops=dict(arrowstyle="->", color=color, lw=1.2),
            bbox=dict(boxstyle="round,pad=0.25", facecolor='#FFFFFF', edgecolor=color, alpha=0.9)
        )

    ax.set_xlabel("Recall (Tingkat Deteksi Fraud / Sensitivitas)", fontsize=11, fontweight='bold', color='#1E293B')
    ax.set_ylabel("Precision (Akurasi Prediksi Fraud / PPV)", fontsize=11, fontweight='bold', color='#1E293B')
    ax.set_title("EXP21: Kurva Kontinu Precision-Recall (PR Curve)\n"
                 "(Trade-off Kritis Deteksi Penipuan pada Kelas Sangat Tidak Seimbang)",
                 fontsize=13, fontweight='bold', pad=15, color='#0F172A')
    
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(color='#E2E8F0', linestyle='--', linewidth=0.8)
    ax.legend(loc='lower left', fontsize=10, framealpha=0.95, facecolor='#FFFFFF', edgecolor='#CBD5E1')

    plt.tight_layout()
    out_path = out_dir / "chart10_precision_recall_curve.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart11_roc_curves(results, predictions, out_dir: Path):
    """
    Chart 11: Kurva Kontinu Receiver Operating Characteristic (ROC Curve) Berlekuk
    Menampilkan kurva global beserta inset pembesaran pada False Positive Rate rendah (< 5%).
    """
    fig, ax = plt.subplots(figsize=(10, 7.2), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#F8FAFC')

    methods = ['uniform', 'topology', 'importance']
    
    # Chance line
    ax.plot([0, 1], [0, 1], color='#94A3B8', linestyle='--', linewidth=1.5, label='Random Chance (AUC = 0.5000)')

    for m in methods:
        probs = predictions[m]['probs']
        labels = predictions[m]['labels']
        tau_star = results[m]['metrics_optimal_threshold']['threshold']
        roc_auc_val = results[m]['metrics_threshold_independent']['roc_auc']
        
        fpr, tpr, _ = roc_curve(labels, probs)
        
        color = METHOD_COLORS[m]
        name = METHOD_NAMES[m]
        ax.plot(fpr, tpr, color=color, linewidth=2.6, 
                label=f'{name} (ROC-AUC = {roc_auc_val:.4f})')
        
        # Titik tau*
        preds_tau = (probs >= tau_star).astype(int)
        tp = np.sum((preds_tau == 1) & (labels == 1))
        fp = np.sum((preds_tau == 1) & (labels == 0))
        fn = np.sum((preds_tau == 0) & (labels == 1))
        tn = np.sum((preds_tau == 0) & (labels == 0))
        pt_tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        pt_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        
        ax.plot(pt_fpr, pt_tpr, marker='*', markersize=14, color=color, 
                markeredgecolor='#0F172A', markeredgewidth=1.2, zorder=5)

    ax.set_xlabel("False Positive Rate (FPR / False Alarm Rate)", fontsize=11, fontweight='bold', color='#1E293B')
    ax.set_ylabel("True Positive Rate (TPR / Hit Rate / Recall)", fontsize=11, fontweight='bold', color='#1E293B')
    ax.set_title("EXP21: Kurva Kontinu ROC (Receiver Operating Characteristic)\n"
                 "(Kemampuan Separasi Global Model GraphSAGE + XGBoost)",
                 fontsize=13, fontweight='bold', pad=15, color='#0F172A')
    
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(color='#E2E8F0', linestyle='--', linewidth=0.8)
    ax.legend(loc='lower right', fontsize=10, framealpha=0.95, facecolor='#FFFFFF', edgecolor='#CBD5E1')

    # Inset zoom pada False Positive Rate rendah [0, 0.05]
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    axins = inset_axes(ax, width="42%", height="40%", loc='center right', borderpad=2)
    axins.set_facecolor('#F8FAFC')
    
    for m in methods:
        probs = predictions[m]['probs']
        labels = predictions[m]['labels']
        fpr, tpr, _ = roc_curve(labels, probs)
        axins.plot(fpr, tpr, color=METHOD_COLORS[m], linewidth=2.2)
    
    axins.plot([0, 1], [0, 1], color='#94A3B8', linestyle='--', linewidth=1.2)
    axins.set_xlim(0.0, 0.03)
    axins.set_ylim(0.85, 1.00)
    axins.set_title("Zoom FPR Rendah (< 3%)", fontsize=8.5, fontweight='bold', color='#334155')
    axins.grid(color='#E2E8F0', linestyle=':', linewidth=0.8)
    axins.tick_params(axis='both', which='major', labelsize=7.5)

    out_path = out_dir / "chart11_roc_curve.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart12_learning_curves_loss(histories, out_dir: Path):
    """
    Chart 12: Kurva Pembelajaran Loss (Training vs Validation Loss per Epoch)
    Menampilkan dinamika konvergensi dan kestabilan optimasi GNN pada 10 Epoch.
    """
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), dpi=300, sharey=True)
    fig.patch.set_facecolor('#FFFFFF')

    methods = ['uniform', 'topology', 'importance']
    
    for idx, m in enumerate(methods):
        ax = axes[idx]
        ax.set_facecolor('#F8FAFC')
        hist = histories[m]['gnn_history']
        epochs = hist['epochs']
        train_loss = hist['train_loss']
        val_loss = hist['val_loss']
        color = METHOD_COLORS[m]
        
        ax.plot(epochs, train_loss, marker='o', linewidth=2.4, markersize=6, 
                color=color, label='Training Loss')
        ax.plot(epochs, val_loss, marker='s', linewidth=2.2, markersize=6, 
                linestyle='--', color='#EF4444', label='Validation Loss')
        
        # Area selisih
        ax.fill_between(epochs, train_loss, val_loss, color=color, alpha=0.10)
        
        # Highlight Best Checkpoint jika tersedia
        best_ep = histories[m].get('best_epoch', None)
        if best_ep is not None and best_ep in epochs:
            ax.axvline(best_ep, color='#D97706', linestyle=':', linewidth=1.8, label=f'Best Checkpoint (Ep {best_ep})')
        
        ax.set_title(f"{METHOD_NAMES[m]}", fontsize=12, fontweight='bold', color='#1E293B', pad=10)
        ax.set_xlabel("Epoch Pelatihan GNN", fontsize=10.5, fontweight='bold', color='#334155')
        if idx == 0:
            ax.set_ylabel("Weighted BCE Loss", fontsize=11, fontweight='bold', color='#334155')
            
        ax.set_xticks(epochs)
        ax.grid(color='#E2E8F0', linestyle='--', linewidth=0.8)
        ax.legend(loc='upper right', fontsize=9.0, framealpha=0.95, facecolor='#FFFFFF')

    fig.suptitle("EXP21: Kurva Konvergensi Loss Pelatihan GNN (Training vs Validation)\n"
                 "(10 Epochs GraphSAGE dengan Pembobotan Pos_Weight)",
                 fontsize=13.5, fontweight='bold', y=1.03, color='#0F172A')

    plt.tight_layout()
    out_path = out_dir / "chart12_learning_curves_loss.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart13_learning_curves_accuracy(histories, out_dir: Path):
    """
    Chart 13: Kurva Pembelajaran Akurasi (Training vs Validation Accuracy per Epoch)
    """
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), dpi=300, sharey=True)
    fig.patch.set_facecolor('#FFFFFF')

    methods = ['uniform', 'topology', 'importance']
    
    for idx, m in enumerate(methods):
        ax = axes[idx]
        ax.set_facecolor('#F8FAFC')
        hist = histories[m]['gnn_history']
        epochs = hist['epochs']
        train_acc = [acc * 100 for acc in hist['train_accuracy']]
        val_acc = [acc * 100 for acc in hist['val_accuracy']]
        color = METHOD_COLORS[m]
        
        ax.plot(epochs, train_acc, marker='o', linewidth=2.4, markersize=6, 
                color=color, label='Training Accuracy (%)')
        ax.plot(epochs, val_acc, marker='^', linewidth=2.2, markersize=6, 
                linestyle='--', color='#EF4444', label='Val Acc @ 0.5 (%)')
        
        # Tampilkan juga Akurasi Terkalibrasi @ tau* jika tersedia di v4
        if 'val_calibrated_accuracy' in hist and len(hist['val_calibrated_accuracy']) == len(epochs):
            val_acc_cal = [acc * 100 for acc in hist['val_calibrated_accuracy']]
            ax.plot(epochs, val_acc_cal, marker='d', linewidth=2.0, markersize=5,
                    linestyle='-.', color='#059669', label='Val Acc @ tau* (%)')
        
        ax.set_title(f"{METHOD_NAMES[m]}", fontsize=12, fontweight='bold', color='#1E293B', pad=10)
        ax.set_xlabel("Epoch Pelatihan GNN", fontsize=10.5, fontweight='bold', color='#334155')
        if idx == 0:
            ax.set_ylabel("Akurasi (%)", fontsize=11, fontweight='bold', color='#334155')
            
        ax.set_xticks(epochs)
        ax.grid(color='#E2E8F0', linestyle='--', linewidth=0.8)
        ax.legend(loc='lower right', fontsize=8.5, framealpha=0.95, facecolor='#FFFFFF')

    fig.suptitle("EXP21: Kurva Akurasi Pelatihan GNN Tiap Epoch (Training vs Validation)",
                 fontsize=13.5, fontweight='bold', y=1.03, color='#0F172A')

    plt.tight_layout()
    out_path = out_dir / "chart13_learning_curves_accuracy.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart14_learning_curves_multimetric(histories, out_dir: Path):
    """
    Chart 14: Kurva Multi-Metrik Dinamis per Epoch (F1-Score, Recall, Precision, AUCPR)
    Menampilkan perbandingan komparatif evolusi performa ketiga metode sampling pada Validation Set.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')

    methods = ['uniform', 'topology', 'importance']
    metrics_to_plot = [
        ('val_f1', 'F1-Score (Validation)', axes[0, 0]),
        ('val_recall', 'Recall / Hit Rate (Validation)', axes[0, 1]),
        ('val_precision', 'Precision / PPV (Validation)', axes[1, 0]),
        ('val_auprc', 'AUCPR / Average Precision (Validation)', axes[1, 1])
    ]

    for metric_key, title, ax in metrics_to_plot:
        ax.set_facecolor('#F8FAFC')
        for m in methods:
            hist = histories[m]['gnn_history']
            epochs = hist['epochs']
            values = hist[metric_key]
            ax.plot(epochs, values, marker='o', linewidth=2.4, markersize=5.5,
                    color=METHOD_COLORS[m], label=METHOD_NAMES[m])
            
        ax.set_title(title, fontsize=11.5, fontweight='bold', color='#1E293B', pad=10)
        ax.set_xlabel("Epoch Pelatihan GNN", fontsize=10, fontweight='bold', color='#475569')
        ax.set_ylabel("Nilai Metrik", fontsize=10, fontweight='bold', color='#475569')
        ax.set_xticks(epochs)
        ax.grid(color='#E2E8F0', linestyle='--', linewidth=0.8)
        ax.legend(loc='best', fontsize=9, framealpha=0.95, facecolor='#FFFFFF')

    fig.suptitle("EXP21: Dinamika Multi-Metrik Evaluasi Tiap Epoch pada Validation Set\n"
                 "(Evolusi F1, Recall, Precision, dan AUCPR Selama Pelatihan GraphSAGE)",
                 fontsize=13.5, fontweight='bold', y=0.995, color='#0F172A')

    plt.tight_layout()
    out_path = out_dir / "chart14_learning_curves_multimetric.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart15_xgboost_boosting_dynamics(histories, out_dir: Path):
    """
    Chart 15: Dinamika Boosting XGBoost Tiap Iterasi / Pohon (1-100 Trees)
    Menampilkan Logloss, AUCPR, dan ROC-AUC pada Train dan Validation set.
    """
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')

    methods = ['uniform', 'topology', 'importance']
    metrics_info = [
        ('val_logloss', 'Log-Loss (Validation)', axes[0], 'Log Loss'),
        ('val_aucpr', 'AUCPR (Validation)', axes[1], 'Area under PR Curve'),
        ('val_auc', 'ROC-AUC (Validation)', axes[2], 'Area under ROC Curve')
    ]

    for key, title, ax, ylabel in metrics_info:
        ax.set_facecolor('#F8FAFC')
        for m in methods:
            xgb_h = histories[m]['xgb_history']
            values = xgb_h.get(key, [])
            iters = np.arange(1, len(values) + 1)
            ax.plot(iters, values, linewidth=2.2, color=METHOD_COLORS[m], label=METHOD_NAMES[m])
            
        ax.set_title(title, fontsize=11.5, fontweight='bold', color='#1E293B', pad=10)
        ax.set_xlabel("Iterasi Boosting (Tree Estimator)", fontsize=10, fontweight='bold', color='#475569')
        ax.set_ylabel(ylabel, fontsize=10, fontweight='bold', color='#475569')
        ax.grid(color='#E2E8F0', linestyle='--', linewidth=0.8)
        ax.legend(loc='best', fontsize=9, framealpha=0.95, facecolor='#FFFFFF')

    fig.suptitle("EXP21: Dinamika Konvergensi XGBoost Classifier per Iterasi Boosting\n"
                 "(100 Pohon Keputusan Dilatih pada Representasi Gabungan GraphSAGE + Fitur Tabular)",
                 fontsize=13.5, fontweight='bold', y=1.03, color='#0F172A')

    plt.tight_layout()
    out_path = out_dir / "chart15_xgboost_boosting_curves.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def exponential_moving_average(values, alpha=0.02):
    """Menghitung Exponential Moving Average untuk menghaluskan tren kurva."""
    if len(values) == 0:
        return np.array([])
    ema = np.empty_like(values, dtype=np.float64)
    ema[0] = values[0]
    for i in range(1, len(values)):
        ema[i] = alpha * values[i] + (1.0 - alpha) * ema[i - 1]
    return ema

def plot_chart16_step_loss(histories, out_dir: Path):
    """
    Chart 16: Kurva Konvergensi Loss GNN per-Step / per-Batch Iterasi
    Menampilkan Raw Mini-Batch Loss dan Smoothed Trendline (EMA) untuk 3 metode sampling.
    """
    methods = ['uniform', 'topology', 'importance']
    has_step_data = any('gnn_step_history' in histories.get(m, {}) and len(histories[m]['gnn_step_history'].get('steps', [])) > 0 for m in methods)
    if not has_step_data:
        print("[INFO] Data gnn_step_history belum tersedia untuk Chart 16. Melewati pembuatan chart.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')

    for idx, m in enumerate(methods):
        ax = axes[idx]
        ax.set_facecolor('#F8FAFC')
        step_hist = histories.get(m, {}).get('gnn_step_history', {})
        steps = np.array(step_hist.get('steps', []))
        loss = np.array(step_hist.get('train_loss', []))
        epochs_arr = np.array(step_hist.get('epochs', []))

        if len(steps) > 0 and len(loss) > 0:
            ema_loss = exponential_moving_average(loss, alpha=0.02)
            # Plot raw batch loss (semi-transparent)
            ax.plot(steps, loss, color=METHOD_COLORS[m], alpha=0.18, linewidth=0.8, label='Raw Batch Loss')
            # Plot smoothed EMA trendline (thick line)
            ax.plot(steps, ema_loss, color=METHOD_COLORS[m], linewidth=2.6, label='Smoothed Trendline (EMA)')

            # Anotasi batas Epoch
            if len(epochs_arr) > 0:
                epoch_changes = np.where(np.diff(epochs_arr) > 0)[0]
                for ep_idx in epoch_changes:
                    ax.axvline(x=steps[ep_idx], color='#94A3B8', linestyle=':', alpha=0.5, linewidth=0.9)

        ax.set_title(METHOD_NAMES[m], fontsize=12, fontweight='bold', color='#1E293B', pad=10)
        ax.set_xlabel("Langkah Pelatihan Mini-Batch (Training Steps)", fontsize=10, fontweight='bold', color='#475569')
        ax.set_ylabel("Weighted BCE Loss", fontsize=10, fontweight='bold', color='#475569')
        ax.grid(color='#E2E8F0', linestyle='--', linewidth=0.8)
        ax.legend(loc='upper right', fontsize=9, framealpha=0.95, facecolor='#FFFFFF')

    fig.suptitle("EXP21: Kurva Konvergensi Loss GNN per-Langkah Mini-Batch (Training Steps)\n"
                 "(Dinamika Pembelajaran Langkah demi Langkah GraphSAGE dengan Exponential Moving Average)",
                 fontsize=13.5, fontweight='bold', y=1.03, color='#0F172A')

    plt.tight_layout()
    out_path = out_dir / "chart16_step_loss.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def plot_chart17_step_accuracy(histories, out_dir: Path):
    """
    Chart 17: Kurva Konvergensi Akurasi GNN per-Step / per-Batch Iterasi
    Menampilkan Raw Batch Accuracy dan Smoothed Trendline (EMA) untuk 3 metode sampling.
    """
    methods = ['uniform', 'topology', 'importance']
    has_step_data = any('gnn_step_history' in histories.get(m, {}) and len(histories[m]['gnn_step_history'].get('steps', [])) > 0 for m in methods)
    if not has_step_data:
        print("[INFO] Data gnn_step_history belum tersedia untuk Chart 17. Melewati pembuatan chart.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), dpi=300)
    fig.patch.set_facecolor('#FFFFFF')

    for idx, m in enumerate(methods):
        ax = axes[idx]
        ax.set_facecolor('#F8FAFC')
        step_hist = histories.get(m, {}).get('gnn_step_history', {})
        steps = np.array(step_hist.get('steps', []))
        acc = np.array(step_hist.get('train_accuracy', [])) * 100.0  # Ke persen
        epochs_arr = np.array(step_hist.get('epochs', []))

        if len(steps) > 0 and len(acc) > 0:
            ema_acc = exponential_moving_average(acc, alpha=0.02)
            # Plot raw batch accuracy
            ax.plot(steps, acc, color=METHOD_COLORS[m], alpha=0.18, linewidth=0.8, label='Raw Batch Accuracy')
            # Plot smoothed EMA trendline
            ax.plot(steps, ema_acc, color=METHOD_COLORS[m], linewidth=2.6, label='Smoothed Trendline (EMA)')

            # Anotasi batas Epoch
            if len(epochs_arr) > 0:
                epoch_changes = np.where(np.diff(epochs_arr) > 0)[0]
                for ep_idx in epoch_changes:
                    ax.axvline(x=steps[ep_idx], color='#94A3B8', linestyle=':', alpha=0.5, linewidth=0.9)

        ax.set_title(METHOD_NAMES[m], fontsize=12, fontweight='bold', color='#1E293B', pad=10)
        ax.set_xlabel("Langkah Pelatihan Mini-Batch (Training Steps)", fontsize=10, fontweight='bold', color='#475569')
        ax.set_ylabel("Batch Training Accuracy (%)", fontsize=10, fontweight='bold', color='#475569')
        ax.grid(color='#E2E8F0', linestyle='--', linewidth=0.8)
        ax.legend(loc='lower right', fontsize=9, framealpha=0.95, facecolor='#FFFFFF')

    fig.suptitle("EXP21: Kurva Konvergensi Akurasi GNN per-Langkah Mini-Batch (Training Steps)\n"
                 "(Evolusi Akurasi Mini-Batch Langkah demi Langkah dengan Exponential Moving Average)",
                 fontsize=13.5, fontweight='bold', y=1.03, color='#0F172A')

    plt.tight_layout()
    out_path = out_dir / "chart17_step_accuracy.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SAVED] {out_path.name}")

def main():
    parser = argparse.ArgumentParser(description="EXP21: Visualisasi Komprehensif Hasil Eksperimen")
    parser.add_argument('--result-dir', type=str, default='../../result/exp21', help='Direktori artefak hasil exp21')
    parser.add_argument('--output-dir', type=str, default='../../result/exp21/plots', help='Direktori penyimpanan plot gambar')
    args = parser.parse_args()
    
    result_dir = Path(args.result_dir).resolve()
    output_dir = Path(args.output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n=======================================================")
    print(f"  MEMULAI GENERASI VISUALISASI EXP21")
    print(f"  • Direktori Data : {result_dir}")
    print(f"  • Direktori Plot : {output_dir}")
    print(f"=======================================================\n")
    
    results, window_stats = load_data(result_dir)
    
    # 1. Primary Metrics
    plot_chart1_primary_metrics(results, output_dir)
    # 2. Threshold Calibration Effect
    plot_chart2_threshold_calibration(results, output_dir)
    # 3. Confusion Matrix
    plot_chart3_confusion_matrix(results, window_stats, output_dir)
    # 4. Banking Operations (TP@500 & Precision@500)
    plot_chart4_banking_operations(results, window_stats, output_dir)
    # 5. Lift@1%
    plot_chart5_lift_percentile(results, output_dir)
    # 6. Runtime Breakdown
    plot_chart6_runtime_breakdown(results, output_dir)
    # 7. Temporal Timeline
    plot_chart7_temporal_window_timeline(window_stats, output_dir)
    # 8. Data Distribution & Split Characteristics
    plot_chart8_data_distribution(window_stats, output_dir)
    # 9. Multi-criteria Radar
    plot_chart9_radar_synthesis(results, output_dir)

    print(f"\n--- [Generasi Kurva Kontinu & Learning Curves (Chart 10 - 17)] ---")
    predictions = load_predictions(result_dir)
    if predictions is not None:
        plot_chart10_precision_recall_curves(results, predictions, window_stats, output_dir)
        plot_chart11_roc_curves(results, predictions, output_dir)
    else:
        print("[INFO] File eksp21_{method}_predictions.npz belum tersedia di result dir.")
        print("       Jalankan ulang training di Kaggle untuk menghasilkan kurva PR & ROC kontinu.")

    histories = load_histories(result_dir)
    if histories is not None:
        plot_chart12_learning_curves_loss(histories, output_dir)
        plot_chart13_learning_curves_accuracy(histories, output_dir)
        plot_chart14_learning_curves_multimetric(histories, output_dir)
        plot_chart15_xgboost_boosting_dynamics(histories, output_dir)
        plot_chart16_step_loss(histories, output_dir)
        plot_chart17_step_accuracy(histories, output_dir)
    else:
        print("[INFO] File eksp21_{method}_history.json belum tersedia di result dir.")
        print("       Jalankan ulang training di Kaggle untuk menghasilkan kurva Loss, Acc, dan Multi-Metrik per-epoch.")
    
    print(f"\n=======================================================")
    print(f"  [SUCCESS] PROSES GENERASI VISUALISASI SELESAI!")
    print(f"  Cek hasil visualisasi di: {output_dir}")
    print(f"=======================================================\n")

if __name__ == '__main__':
    main()



In [ ]:
%%writefile main.py
import argparse
import time
import json
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from sklearn.metrics import (
    f1_score, recall_score, precision_score, 
    accuracy_score, roc_auc_score, average_precision_score, 
    confusion_matrix
)
from xgboost import XGBClassifier

from config import load_config
from data_preprocessing import load_and_preprocess_data, get_temporal_window_split
from graph_builder import build_hetero_graph, graph_level_undersample
from samplers import get_neighbor_loader
from model import HeteroGraphSAGE, manual_weighted_bce, manual_unweighted_bce

def extract_features(model, loader, device, total_samples, orig_feat_dim, emb_dim):
    """
    Mengekstrak embedding GNN dan menggabungkannya dengan fitur orisinal transaksi.
    Menggunakan pre-alokasi memori numpy untuk efisiensi RAM maksimal.
    """
    model.eval()
    final_features = np.empty((total_samples, orig_feat_dim + emb_dim), dtype=np.float32)
    all_targets = np.empty((total_samples,), dtype=np.int64)
    
    start_time = time.time()
    current_idx = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            
            embeddings = model(
                batch.x_dict, 
                batch.edge_index_dict, 
                num_nodes_dict=num_nodes_dict, 
                return_embeddings=True
            )
            
            batch_size = batch['transaction'].batch_size
            next_idx = min(current_idx + batch_size, total_samples)
            actual_count = next_idx - current_idx
            
            final_features[current_idx:next_idx, :orig_feat_dim] = batch['transaction'].x[:actual_count].cpu().numpy()
            final_features[current_idx:next_idx, orig_feat_dim:] = embeddings[:actual_count].cpu().numpy()
            all_targets[current_idx:next_idx] = batch['transaction'].y[:actual_count].cpu().numpy()
            
            current_idx = next_idx
            
    inference_time = time.time() - start_time
    
    if current_idx < total_samples:
        final_features = final_features[:current_idx]
        all_targets = all_targets[:current_idx]
        
    return final_features, all_targets, inference_time

def find_optimal_threshold(val_probs, val_labels, thresholds=None):
    """
    Mencari threshold optimal tau* yang memaksimalkan F1-score pada Validation set.
    """
    if thresholds is None:
        thresholds = np.arange(0.01, 0.51, 0.005)
        
    best_tau = 0.5
    best_f1 = 0.0
    best_metrics = {}
    
    for tau in thresholds:
        preds = (val_probs >= tau).astype(int)
        f1 = f1_score(val_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_tau = float(tau)
            best_metrics = {
                "tau": float(tau),
                "val_f1": float(f1),
                "val_precision": float(precision_score(val_labels, preds, zero_division=0)),
                "val_recall": float(recall_score(val_labels, preds, zero_division=0))
            }
            
    if best_f1 == 0.0:
        best_tau = 0.5
        best_metrics = {
            "tau": 0.5,
            "val_f1": 0.0,
            "val_precision": 0.0,
            "val_recall": 0.0
        }
        
    return best_tau, best_metrics

def evaluate_gnn(model, loader, device, pos_weight):
    """
    Mengevaluasi model GNN pada loader tertentu.
    Menghitung loss terbobot, loss unweighted (standar), metrik @ 0.5, metrik @ tau* (terkalibrasi), auprc, dan roc_auc.
    """
    model.eval()
    total_loss = 0.0
    total_loss_unw = 0.0
    total_samples = 0
    all_probs = []
    all_targets = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            probs = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            batch_size_current = batch['transaction'].batch_size
            probs = probs[:batch_size_current]
            targets = batch['transaction'].y[:batch_size_current]
            loss = manual_weighted_bce(probs, targets, pos_weight)
            loss_unw = manual_unweighted_bce(probs, targets)
            
            total_loss += loss.item() * batch_size_current
            total_loss_unw += loss_unw.item() * batch_size_current
            total_samples += batch_size_current
            all_probs.append(probs.cpu())
            all_targets.append(targets.cpu())
            
    all_probs = torch.cat(all_probs).numpy().ravel()
    all_targets = torch.cat(all_targets).numpy().ravel()
    avg_loss = total_loss / max(1, total_samples)
    avg_loss_unw = total_loss_unw / max(1, total_samples)
    
    # 1. Metrik @ Baseline Fixed Threshold 0.5
    preds_05 = (all_probs >= 0.5).astype(int)
    acc_05 = float(accuracy_score(all_targets, preds_05))
    f1_05 = float(f1_score(all_targets, preds_05, zero_division=0))
    prec_05 = float(precision_score(all_targets, preds_05, zero_division=0))
    rec_05 = float(recall_score(all_targets, preds_05, zero_division=0))
    
    # 2. Metrik Bebas Threshold
    aucpr = float(average_precision_score(all_targets, all_probs)) if len(np.unique(all_targets)) > 1 else 0.0
    roc_auc = float(roc_auc_score(all_targets, all_probs)) if len(np.unique(all_targets)) > 1 else 0.0
    
    # 3. Metrik @ Optimal Threshold tau* (Mengatasi Paradoks Metrik & Miskalibrasi)
    best_tau, tau_stats = find_optimal_threshold(all_probs, all_targets)
    preds_tau = (all_probs >= best_tau).astype(int)
    acc_tau = float(accuracy_score(all_targets, preds_tau))
    
    return {
        'loss': float(avg_loss),
        'loss_unweighted': float(avg_loss_unw),
        'accuracy': acc_05,
        'f1_score': f1_05,
        'precision': prec_05,
        'recall': rec_05,
        'auprc': aucpr,
        'roc_auc': roc_auc,
        'optimal_tau': float(best_tau),
        'calibrated_accuracy': acc_tau,
        'calibrated_f1': float(tau_stats['val_f1']),
        'calibrated_precision': float(tau_stats['val_precision']),
        'calibrated_recall': float(tau_stats['val_recall'])
    }

def main():
    parser = argparse.ArgumentParser(description="Eksperimen 21: Temporal Window Alignment & Inductive Setup")
    parser.add_argument('--config', type=str, default='config.reference.yaml', help='Path ke file konfigurasi YAML')
    parser.add_argument('--sampling', type=str, choices=['uniform', 'topology', 'importance'], default=None, help='Metode neighbor sampling')
    parser.add_argument('--reference-date', type=str, default=None, help='Override tanggal T0 window (format: YYYY-MM-DD)')
    parser.add_argument('--nrows', type=int, default=None, help='Limit baris data untuk debugging lokal')
    parser.add_argument('--start-year', type=int, default=None, help='Override filter tahun awal dataset')
    args = parser.parse_args()
    
    total_pipeline_start = time.time()
    
    # 1. Load Konfigurasi
    cfg = load_config(args.config)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*75}")
    print(f"  EXP21: TEMPORAL WINDOW ALIGNMENT & INDUCTIVE ROLLING WINDOW SETUP")
    print(f"{'='*75}")
    print(f"Device               : {device}")
    
    sampling_method = args.sampling if args.sampling is not None else cfg['sampling']['method']
    print(f"Sampling Method      : {sampling_method}")
    
    # Override time_window parameters jika disediakan di CLI
    tw_cfg = cfg.get('time_window', {})
    reference_date = args.reference_date if args.reference_date is not None else tw_cfg.get('reference_date', '2018-01-01')
    train_days = tw_cfg.get('train_days', 300)
    val_days = tw_cfg.get('val_days', 30)
    gap_days = tw_cfg.get('gap_days', 8)
    test_days = tw_cfg.get('test_days', 60)
    
    start_year = args.start_year if args.start_year is not None else cfg['data'].get('start_year', 2016)
    dataset_path = cfg['data']['transactions']
    nrows = args.nrows if args.nrows is not None else cfg['data'].get('max_rows', None)
    
    # Siapkan direktori output
    model_dir = Path(cfg.get('output', {}).get('model_dir', '../../model/exp21'))
    result_dir = Path(cfg.get('output', {}).get('result_dir', '../../result/exp21'))
    model_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    # 2. Preprocessing Data & Filter Temporal 2016-2020
    print("\n--- [Langkah 1: Preprocessing Data (2016-2020)] ---")
    data_prep_start = time.time()
    df, feature_cols = load_and_preprocess_data(dataset_path, start_year=start_year, nrows=nrows)
    data_prep_time = time.time() - data_prep_start
    print(f"Data preprocessing selesai dalam {data_prep_time:.2f} detik. Total baris: {len(df)}")
    
    # 3. Pembagian Temporal Window (300d train, 30d val, 8d gap, 60d test)
    print("\n--- [Langkah 2: Pembagian Jendela Temporal SHINE FRAUD II] ---")
    train_df, val_df, test_df, gap_df, window_stats = get_temporal_window_split(
        df, 
        reference_date=reference_date,
        train_days=train_days,
        val_days=val_days,
        gap_days=gap_days,
        test_days=test_days
    )
    
    print("\nStatistik Partisi Jendela Temporal:")
    for split_key, sp in window_stats['splits'].items():
        print(f"  • {sp['name']:12s}: {sp['num_transactions']:8d} transaksi | "
              f"{sp['num_fraud']:5d} fraud ({sp['fraud_rate_pct']:6.3f}%) | "
              f"Rentang: {sp['start_date'][:10]} s/d {sp['end_date'][:10]}")
              
    # Simpan window_statistics.json
    stats_path = result_dir / "window_statistics.json"
    with open(stats_path, 'w') as f:
        json.dump(window_stats, f, indent=4)
    print(f"Statistik jendela waktu tersimpan ke: {stats_path}")
    
    if len(train_df) == 0 or len(test_df) == 0:
        raise ValueError("Dataframe Train atau Test kosong! Periksa kembali parameter reference_date atau nrows.")

    # 4. Undersampling (Default: 1.0 = Tanpa Undersampling)
    undersample_ratio = cfg['sampling'].get('undersampling_ratio', 1.0)
    train_df_sampled = graph_level_undersample(train_df, ratio=undersample_ratio)

    # 5. Konstruksi Graf Heterogen (PyG)
    print("\n--- [Langkah 3: Membangun Graf Train] ---")
    train_data = build_hetero_graph(train_df_sampled, feature_cols)
    
    batch_size = cfg['training'].get('batch_size', 1024)
    num_neighbors = cfg['training'].get('num_neighbors', [25, 10])
    
    train_loader = get_neighbor_loader(
        train_data, 
        batch_size=batch_size, 
        num_neighbors=num_neighbors,
        method=sampling_method, 
        shuffle=True
    )
    
    # 6. Definisi Arsitektur Model GNN (HeteroGraphSAGE)
    feature_dims = {
        'transaction': len(feature_cols),
        'user': 2,
        'merchant': 2
    }
    hidden_channels = cfg['model'].get('hidden_channels', 256)
    out_channels = cfg['model'].get('out_channels', 1)
    num_layers = cfg['model'].get('num_layers', 2)
    aggr = cfg['model'].get('aggr', 'mean')
    dropout = float(cfg['model'].get('dropout', 0.2))
    
    model = HeteroGraphSAGE(
        hidden_channels=hidden_channels,
        out_channels=out_channels,
        num_layers=num_layers,
        data_metadata=train_data.metadata(),
        feature_dims=feature_dims,
        aggr=aggr,
        dropout=dropout
    ).to(device)
    
    num_pos = int(train_data['transaction'].y.sum().item())
    num_neg = int(train_data['transaction'].num_nodes - num_pos)
    raw_pos_weight = float(num_neg / (num_pos + 1e-7))
    
    # Strategi peredaman bobot kelas (Anti-exploding loss)
    loss_weight_strategy = cfg.get('training', {}).get('loss_weight_strategy', 'sqrt')
    if loss_weight_strategy == 'sqrt':
        eff_pos_weight = float(np.sqrt(raw_pos_weight))
    elif loss_weight_strategy == 'clipped':
        max_pw = float(cfg.get('training', {}).get('max_pos_weight', 50.0))
        eff_pos_weight = float(min(raw_pos_weight, max_pw))
    else:  # 'full'
        eff_pos_weight = float(raw_pos_weight)
        
    pos_weight = torch.tensor([eff_pos_weight]).to(device)
    print(f"GNN Class-Weight pos_weight ({loss_weight_strategy}): {pos_weight.item():.2f} (raw ratio: {raw_pos_weight:.2f})")
    print(f"GNN Architecture Regularization: Dropout = {dropout:.2f}")
    
    lr = float(cfg['training'].get('learning_rate', 0.001))
    weight_decay = float(cfg['training'].get('weight_decay', 1e-4))
    label_smoothing = float(cfg.get('training', {}).get('label_smoothing', 0.05))
    grad_clip_norm = float(cfg.get('training', {}).get('grad_clip_norm', 1.0))
    warmup_epochs = int(cfg.get('training', {}).get('warmup_epochs', 1))
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    print(f"Optimizer: Adam (lr={lr}, weight_decay={weight_decay}) | Label Smoothing: {label_smoothing} | Grad Clip: {grad_clip_norm} | Warmup Epochs: {warmup_epochs}")

    # Siapkan Validation Data & Loader (dibangun sekali untuk evaluasi per-epoch & ekstraksi fitur Phase 2)
    print("Membangun Graf Validation untuk evaluasi per-epoch...")
    val_graph_df = pd.concat([train_df, val_df], ignore_index=True)
    val_data = build_hetero_graph(val_graph_df, feature_cols)
    val_node_indices = torch.arange(len(train_df), len(val_graph_df))
    val_loader = get_neighbor_loader(
        val_data,
        batch_size=4096,
        num_neighbors=num_neighbors,
        method=sampling_method,
        shuffle=False,
        input_nodes=('transaction', val_node_indices)
    )

    # Learning Rate Scheduler (ReduceLROnPlateau berbasis Val AUPRC)
    use_scheduler = cfg.get('training', {}).get('use_lr_scheduler', True)
    if use_scheduler:
        lr_factor = float(cfg.get('training', {}).get('lr_scheduler_factor', 0.5))
        lr_patience = int(cfg.get('training', {}).get('lr_scheduler_patience', 5))
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 
            mode='max', 
            factor=lr_factor, 
            patience=lr_patience,
            min_lr=1e-5
        )
        print(f"LR Scheduler: ReduceLROnPlateau (factor={lr_factor}, patience={lr_patience}, monitor='val_auprc')")
    else:
        scheduler = None
        
    early_stopping_patience = int(cfg.get('training', {}).get('early_stopping_patience', 15))

    # 7. Phase 1: Pelatihan GraphSAGE Encoder
    epochs = cfg['training'].get('epochs', 100)
    print(f"\n--- [Phase 1: Training GraphSAGE Encoder ({epochs} Epochs dengan Checkpointing)] ---")
    gnn_history = {
        'epochs': [],
        'learning_rates': [],
        'train_loss': [], 'train_loss_unweighted': [], 'train_accuracy': [], 'train_f1': [], 'train_precision': [], 'train_recall': [], 'train_auprc': [], 'train_roc_auc': [],
        'val_loss': [], 'val_loss_unweighted': [], 'val_accuracy': [], 'val_f1': [], 'val_precision': [], 'val_recall': [], 'val_auprc': [], 'val_roc_auc': [],
        'val_optimal_tau': [], 'val_calibrated_accuracy': [], 'val_calibrated_f1': [], 'val_calibrated_precision': [], 'val_calibrated_recall': []
    }
    
    # Riwayat per-step (per-batch iteration) untuk menangkap kurva konvergensi halus
    gnn_step_history = {
        'steps': [],
        'epochs': [],
        'train_loss': [],
        'train_loss_unweighted': [],
        'train_accuracy': []
    }
    global_step = 0
    log_step_interval = int(cfg.get('training', {}).get('log_step_interval', 1))
    
    best_val_auprc = -1.0
    best_epoch = 1
    best_model_state = None
    patience_counter = 0
    best_model_path = model_dir / f"exp21_{sampling_method}_best_gnn.pt"

    gnn_train_start = time.time()
    num_batches = len(train_loader)
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_loss_unw = 0.0
        total_samples = 0
        epoch_probs = []
        epoch_targets = []
        for batch_idx, batch in enumerate(train_loader):
            # Linear LR Warmup pada epoch pertama untuk mencegah lonjakan sharp-minima awal
            if epoch < warmup_epochs and num_batches > 0:
                warmup_factor = 0.1 + 0.9 * ((batch_idx + 1) / num_batches)
                for pg in optimizer.param_groups:
                    pg['lr'] = lr * warmup_factor

            batch = batch.to(device)
            optimizer.zero_grad()
            
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            probs = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            
            batch_size_current = batch['transaction'].batch_size
            probs = probs[:batch_size_current]
            targets = batch['transaction'].y[:batch_size_current]
            
            loss = manual_weighted_bce(probs, targets, pos_weight, label_smoothing=label_smoothing)
            loss_unw = manual_unweighted_bce(probs, targets)
            loss.backward()
            
            # Gradient clipping untuk mencegah ledakan sampling variance dari super-hubs
            if grad_clip_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
                
            optimizer.step()
            
            global_step += 1
            batch_acc = float(((probs.detach() >= 0.5).int() == targets).float().mean().item())
            
            total_loss += loss.item() * batch_size_current
            total_loss_unw += loss_unw.item() * batch_size_current
            total_samples += batch_size_current
            epoch_probs.append(probs.detach().cpu())
            epoch_targets.append(targets.cpu())
            
            if global_step % log_step_interval == 0:
                gnn_step_history['steps'].append(global_step)
                gnn_step_history['epochs'].append(epoch + 1)
                gnn_step_history['train_loss'].append(float(loss.item()))
                gnn_step_history['train_loss_unweighted'].append(float(loss_unw.item()))
                gnn_step_history['train_accuracy'].append(float(batch_acc))
            
        train_probs_cat = torch.cat(epoch_probs).numpy().ravel()
        train_targets_cat = torch.cat(epoch_targets).numpy().ravel()
        train_loss = total_loss / max(1, total_samples)
        train_loss_unw = total_loss_unw / max(1, total_samples)
        train_preds = (train_probs_cat >= 0.5).astype(int)
        
        train_acc = float(accuracy_score(train_targets_cat, train_preds))
        train_f1 = float(f1_score(train_targets_cat, train_preds, zero_division=0))
        train_prec = float(precision_score(train_targets_cat, train_preds, zero_division=0))
        train_rec = float(recall_score(train_targets_cat, train_preds, zero_division=0))
        train_auprc = float(average_precision_score(train_targets_cat, train_probs_cat)) if len(np.unique(train_targets_cat)) > 1 else 0.0
        train_roc_auc = float(roc_auc_score(train_targets_cat, train_probs_cat)) if len(np.unique(train_targets_cat)) > 1 else 0.0
        
        # Reset learning rate ke nilai nominal base_lr setelah warmup epoch selesai
        if epoch < warmup_epochs:
            for pg in optimizer.param_groups:
                pg['lr'] = lr

        # Evaluasi validation set di akhir epoch
        val_metrics = evaluate_gnn(model, val_loader, device, pos_weight)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Checkpoint Best Model berbasis Validation AUCPR
        if val_metrics['auprc'] > best_val_auprc:
            best_val_auprc = val_metrics['auprc']
            best_epoch = epoch + 1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(model.state_dict(), best_model_path)
            patience_counter = 0
            improved_mark = "★ BEST"
        else:
            patience_counter += 1
            improved_mark = ""
            
        if scheduler is not None and epoch >= (warmup_epochs - 1):
            scheduler.step(val_metrics['auprc'])
        
        # Logging history
        gnn_history['epochs'].append(epoch + 1)
        gnn_history['learning_rates'].append(float(current_lr))
        gnn_history['train_loss'].append(train_loss)
        gnn_history['train_loss_unweighted'].append(train_loss_unw)
        gnn_history['train_accuracy'].append(train_acc)
        gnn_history['train_f1'].append(train_f1)
        gnn_history['train_precision'].append(train_prec)
        gnn_history['train_recall'].append(train_rec)
        gnn_history['train_auprc'].append(train_auprc)
        gnn_history['train_roc_auc'].append(train_roc_auc)
        
        gnn_history['val_loss'].append(val_metrics['loss'])
        gnn_history['val_loss_unweighted'].append(val_metrics['loss_unweighted'])
        gnn_history['val_accuracy'].append(val_metrics['accuracy'])
        gnn_history['val_f1'].append(val_metrics['f1_score'])
        gnn_history['val_precision'].append(val_metrics['precision'])
        gnn_history['val_recall'].append(val_metrics['recall'])
        gnn_history['val_auprc'].append(val_metrics['auprc'])
        gnn_history['val_roc_auc'].append(val_metrics['roc_auc'])
        gnn_history['val_optimal_tau'].append(float(val_metrics['optimal_tau']))
        gnn_history['val_calibrated_accuracy'].append(float(val_metrics['calibrated_accuracy']))
        gnn_history['val_calibrated_f1'].append(float(val_metrics['calibrated_f1']))
        gnn_history['val_calibrated_precision'].append(float(val_metrics['calibrated_precision']))
        gnn_history['val_calibrated_recall'].append(float(val_metrics['calibrated_recall']))
        
        print(f"Epoch {epoch+1:02d}/{epochs:02d} (lr: {current_lr:.6f}) {improved_mark}:")
        print(f"  • Train   | Loss(W): {train_loss:.4f} | Loss(U): {train_loss_unw:.4f} | Acc: {train_acc*100:.2f}% | F1: {train_f1:.4f} | Rec: {train_rec*100:.2f}% | Prec: {train_prec*100:.2f}% | AUCPR: {train_auprc:.4f} | ROC-AUC: {train_roc_auc:.4f}")
        print(f"  • Val@0.5 | Loss(W): {val_metrics['loss']:.4f} | Loss(U): {val_metrics['loss_unweighted']:.4f} | Acc: {val_metrics['accuracy']*100:.2f}% | F1: {val_metrics['f1_score']:.4f} | Rec: {val_metrics['recall']*100:.2f}% | Prec: {val_metrics['precision']*100:.2f}%")
        print(f"  • Val@tau*| tau*: {val_metrics['optimal_tau']:.3f} | Acc: {val_metrics['calibrated_accuracy']*100:.2f}% | F1: {val_metrics['calibrated_f1']:.4f} | Rec: {val_metrics['calibrated_recall']*100:.2f}% | Prec: {val_metrics['calibrated_precision']*100:.2f}% | AUCPR: {val_metrics['auprc']:.4f} | ROC-AUC: {val_metrics['roc_auc']:.4f}")
        
        # Early Stopping check
        if early_stopping_patience > 0 and patience_counter >= early_stopping_patience:
            print(f"\n[EARLY STOPPING] Pelatihan dihentikan lebih awal pada epoch {epoch+1} karena val_auprc tidak membaik selama {early_stopping_patience} epoch berturut-turut.")
            break

    gnn_train_time = time.time() - gnn_train_start
    print(f"Pelatihan GNN selesai dalam {gnn_train_time:.2f} detik.")
    
    # Simpan model GNN state dict terakhir
    gnn_model_path = model_dir / f"exp21_{sampling_method}_gnn.pt"
    torch.save(model.state_dict(), gnn_model_path)
    print(f"Model GNN (last epoch) tersimpan di: {gnn_model_path}")
    print(f"Model GNN (best checkpoint Epoch {best_epoch}, AUCPR={best_val_auprc:.4f}) tersimpan di: {best_model_path}")

    # Pulihkan (restore) bobot model GNN terbaik untuk ekstraksi representasi Phase 2
    if best_model_state is not None:
        print(f"\n[RESTORE CHECKPOINT] Memulihkan bobot terbaik GNN dari Epoch {best_epoch} (Val AUCPR: {best_val_auprc:.4f}) untuk ekstraksi representasi Phase 2...")
        model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})

    # Bebaskan train_loader dan optimizer
    del train_loader, optimizer
    if 'best_model_state' in locals():
        del best_model_state
    gc.collect()
    torch.cuda.empty_cache()

    # 8. Phase 2: Ekstraksi Fitur (Train, Validation, Test)
    print("\n--- [Phase 2: Ekstraksi Fitur Representasi Induktif] ---")
    orig_dim = len(feature_cols)
    emb_dim = hidden_channels
    
    # 8a. Ekstraksi fitur Train
    print(f"Mengekstrak fitur Train ({len(train_df_sampled)} sampel)...")
    train_extract_loader = get_neighbor_loader(
        train_data, 
        batch_size=4096, 
        num_neighbors=num_neighbors, 
        method=sampling_method, 
        shuffle=False
    )
    train_features, train_labels, inf_time_train = extract_features(
        model, train_extract_loader, device,
        total_samples=len(train_df_sampled),
        orig_feat_dim=orig_dim,
        emb_dim=emb_dim
    )
    del train_data, train_extract_loader
    gc.collect()
    torch.cuda.empty_cache()
    
    # 8b. Ekstraksi fitur Validation (menggunakan val_loader yang telah dibangun)
    print(f"Mengekstrak fitur Validation ({len(val_node_indices)} sampel)...")
    val_features, val_labels, inf_time_val_gnn = extract_features(
        model, val_loader, device,
        total_samples=len(val_node_indices),
        orig_feat_dim=orig_dim,
        emb_dim=emb_dim
    )
    del val_data, val_loader, val_graph_df
    gc.collect()
    torch.cuda.empty_cache()

    # 8c. Ekstraksi fitur Test (mencakup Train + Val + Test)
    print(f"Mengekstrak fitur Test ({len(test_df)} sampel)...")
    test_graph_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    test_data = build_hetero_graph(test_graph_df, feature_cols)
    test_node_indices = torch.arange(len(train_df) + len(val_df), len(test_graph_df))
    
    test_loader = get_neighbor_loader(
        test_data,
        batch_size=4096,
        num_neighbors=num_neighbors,
        method=sampling_method,
        shuffle=False,
        input_nodes=('transaction', test_node_indices)
    )
    test_features, test_labels, inf_time_test_gnn = extract_features(
        model, test_loader, device,
        total_samples=len(test_node_indices),
        orig_feat_dim=orig_dim,
        emb_dim=emb_dim
    )
    del test_data, test_loader, test_graph_df, model
    gc.collect()
    torch.cuda.empty_cache()

    # 9. Phase 3: Klasifikasi dengan XGBoost
    print("\n--- [Phase 3: Pelatihan & Prediksi XGBoost Classifier] ---")
    num_neg_xgb = int((train_labels == 0).sum())
    num_pos_xgb = int((train_labels == 1).sum())
    scale_pos_weight = float(num_neg_xgb / num_pos_xgb) if num_pos_xgb > 0 else 1.0
    print(f"XGBoost scale_pos_weight: {scale_pos_weight:.2f}")
    
    cls_cfg = cfg.get('classifier', {})
    xgb_params = {
        'n_estimators': cls_cfg.get('n_estimators', 100),
        'max_depth': cls_cfg.get('max_depth', 5),
        'learning_rate': cls_cfg.get('learning_rate', 0.05),
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'eval_metric': ['logloss', 'aucpr', 'auc']
    }
    
    eval_set = [(train_features, train_labels), (val_features, val_labels)]
    xgb_start = time.time()
    try:
        xgb_model = XGBClassifier(**xgb_params, tree_method='hist', device='cuda')
        xgb_model.fit(train_features, train_labels, eval_set=eval_set, verbose=False)
    except Exception as e:
        print(f"XGBoost GPU fit gagal ({e}), beralih ke CPU...")
        xgb_model = XGBClassifier(**xgb_params, tree_method='hist', n_jobs=-1)
        xgb_model.fit(train_features, train_labels, eval_set=eval_set, verbose=False)
    xgb_train_time = time.time() - xgb_start
    print(f"XGBoost training selesai dalam {xgb_train_time:.2f} detik.")
    xgb_evals_result = xgb_model.evals_result()

    # Simpan model XGBoost
    xgb_model_path = model_dir / f"exp21_{sampling_method}_xgb.json"
    xgb_model.save_model(xgb_model_path)
    print(f"Model XGBoost tersimpan di: {xgb_model_path}")

    # 10. Phase 4: Kalibrasi Threshold Dinamis pada Validation & Evaluasi Test
    print("\n--- [Phase 4: Kalibrasi Threshold Dinamis & Evaluasi Final] ---")
    # Prediksi probabilitas pada Validation
    val_probs = xgb_model.predict_proba(val_features)[:, 1]
    best_tau, val_tuning_stats = find_optimal_threshold(val_probs, val_labels)
    print(f"Optimal Threshold (tau*) dari Validation: {best_tau:.3f} (Val F1: {val_tuning_stats['val_f1']:.4f})")
    
    # Prediksi probabilitas pada Test
    test_pred_start = time.time()
    test_probs = xgb_model.predict_proba(test_features)[:, 1]
    inf_time_xgb = time.time() - test_pred_start
    total_test_inf_time = inf_time_test_gnn + inf_time_xgb

    # Evaluasi @ threshold 0.5 (Default komparasi baseline)
    preds_05 = (test_probs >= 0.5).astype(int)
    f1_05 = f1_score(test_labels, preds_05, zero_division=0)
    recall_05 = recall_score(test_labels, preds_05, zero_division=0)
    precision_05 = precision_score(test_labels, preds_05, zero_division=0)
    
    # Evaluasi @ threshold optimal tau* (Hasil utama)
    preds_tau = (test_probs >= best_tau).astype(int)
    f1_tau = f1_score(test_labels, preds_tau, zero_division=0)
    f1_macro_tau = f1_score(test_labels, preds_tau, average='macro', zero_division=0)
    recall_tau = recall_score(test_labels, preds_tau, zero_division=0)
    precision_tau = precision_score(test_labels, preds_tau, zero_division=0)
    accuracy_tau = accuracy_score(test_labels, preds_tau)
    
    # Metrik sensitivitas & spesifisitas
    cm = confusion_matrix(test_labels, preds_tau)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        gmean = float(np.sqrt(recall_tau * specificity))
    else:
        gmean = 0.0

    # Metrik independen threshold (AUCPR & ROC-AUC)
    roc_auc = roc_auc_score(test_labels, test_probs) if len(np.unique(test_labels)) > 1 else 0.0
    auprc = average_precision_score(test_labels, test_probs) if len(np.unique(test_labels)) > 1 else 0.0

    # Metrik Operasional Perbankan: Lift@1% dan TP@500
    # Lift@1%
    percentile = 0.01
    top_1pct_count = max(1, int(round(len(test_labels) * percentile)))
    top_1pct_idx = np.argsort(test_probs)[::-1][:top_1pct_count]
    precision_1pct = float(test_labels[top_1pct_idx].mean())
    base_fraud_rate = float(test_labels.mean())
    lift_1_percent = (precision_1pct / base_fraud_rate) if base_fraud_rate > 0 else 0.0

    # TP@500
    top_500_count = min(500, len(test_labels))
    top_500_idx = np.argsort(test_probs)[::-1][:top_500_count]
    tp_at_500 = int(test_labels[top_500_idx].sum())
    precision_at_500 = float(tp_at_500 / top_500_count)

    total_pipeline_time = time.time() - total_pipeline_start

    # Tampilkan Ringkasan Hasil
    print("\n" + "="*75)
    print("                      HASIL EVALUASI EKSPERIMEN 21")
    print("="*75)
    print(f"Protokol Waktu      : SHINE FRAUD II (Train: {train_days}d, Val: {val_days}d, Gap: {gap_days}d, Test: {test_days}d)")
    print(f"Sampling Method     : {sampling_method.upper()}")
    print(f"Optimal Threshold   : tau* = {best_tau:.3f} (Validation F1 = {val_tuning_stats['val_f1']:.4f})")
    print("-"*75)
    print(f"Metrik @ Threshold 0.5 (Default Baseline):")
    print(f"  • F1-Score        : {f1_05:.4f}")
    print(f"  • Recall          : {recall_05:.4f} ({recall_05*100:.2f}%)")
    print(f"  • Precision       : {precision_05:.4f} ({precision_05*100:.2f}%)")
    print("-"*75)
    print(f"Metrik @ Optimal Threshold tau* = {best_tau:.3f} (Rekomendasi Utama):")
    print(f"  • F1-Score (Bin)  : {f1_tau:.4f}")
    print(f"  • F1-Score (Macro): {f1_macro_tau:.4f}")
    print(f"  • Recall          : {recall_tau:.4f} ({recall_tau*100:.2f}%)")
    print(f"  • Precision       : {precision_tau:.4f} ({precision_tau*100:.2f}%)")
    print(f"  • Accuracy        : {accuracy_tau:.4f}")
    print(f"  • GMean           : {gmean:.4f}")
    print("-"*75)
    print(f"Metrik Bebas Threshold & Metrik Operasional:")
    print(f"  • AUCPR (AUPRC)   : {auprc:.4f}")
    print(f"  • ROC-AUC         : {roc_auc:.4f}")
    print(f"  • Lift@1%         : {lift_1_percent:.4f}x")
    print(f"  • TP@500          : {tp_at_500}/{top_500_count} ({precision_at_500*100:.2f}%)")
    print("-"*75)
    print(f"Efisiensi Waktu Komputasi:")
    print(f"  • GNN Training    : {gnn_train_time:.2f} detik")
    print(f"  • XGBoost Train   : {xgb_train_time:.2f} detik")
    print(f"  • Test Inference  : {total_test_inf_time:.2f} detik (GNN: {inf_time_test_gnn:.2f}s, XGB: {inf_time_xgb:.2f}s)")
    print(f"  • Total Pipeline  : {total_pipeline_time:.2f} detik ({total_pipeline_time/60:.2f} menit)")
    print("="*75)

    # 11. Simpan Hasil ke Berkas JSON
    result_dict = {
        "experiment_id": "EXP21",
        "experiment_name": "Temporal Window Alignment & Inductive Setup (SHINE FRAUD II - v6 Scaled GraphSAGE)",
        "sampling_method": sampling_method,
        "time_window": {
            "reference_date": reference_date,
            "train_days": train_days,
            "val_days": val_days,
            "gap_days": gap_days,
            "test_days": test_days
        },
        "threshold_calibration": {
            "optimal_tau": float(best_tau),
            "val_f1": float(val_tuning_stats['val_f1']),
            "val_precision": float(val_tuning_stats['val_precision']),
            "val_recall": float(val_tuning_stats['val_recall'])
        },
        "metrics_default_threshold_05": {
            "threshold": 0.5,
            "f1_score": float(f1_05),
            "recall": float(recall_05),
            "precision": float(precision_05)
        },
        "metrics_optimal_threshold": {
            "threshold": float(best_tau),
            "f1_score": float(f1_tau),
            "f1_macro": float(f1_macro_tau),
            "recall": float(recall_tau),
            "precision": float(precision_tau),
            "accuracy": float(accuracy_tau),
            "gmean": float(gmean)
        },
        "metrics_threshold_independent": {
            "auprc": float(auprc),
            "roc_auc": float(roc_auc),
            "lift_1_percent": float(lift_1_percent),
            "tp_at_500": int(tp_at_500),
            "precision_at_500": float(precision_at_500)
        },
        "runtime_seconds": {
            "data_preprocessing": float(data_prep_time),
            "gnn_training": float(gnn_train_time),
            "xgb_training": float(xgb_train_time),
            "gnn_test_inference": float(inf_time_test_gnn),
            "xgb_test_inference": float(inf_time_xgb),
            "total_test_inference": float(total_test_inf_time),
            "total_pipeline": float(total_pipeline_time)
        },
        "hyperparameters": {
            "hidden_channels": hidden_channels,
            "num_layers": num_layers,
            "aggr": aggr,
            "dropout": dropout,
            "gnn_epochs": epochs,
            "gnn_batch_size": batch_size,
            "learning_rate": lr,
            "weight_decay": weight_decay,
            "label_smoothing": label_smoothing,
            "grad_clip_norm": grad_clip_norm,
            "warmup_epochs": warmup_epochs,
            "loss_weight_strategy": loss_weight_strategy,
            "effective_pos_weight": float(pos_weight.item()),
            "best_gnn_epoch": int(best_epoch),
            "best_gnn_val_auprc": float(best_val_auprc),
            "num_neighbors": num_neighbors,
            "xgb_n_estimators": xgb_params['n_estimators'],
            "xgb_max_depth": xgb_params['max_depth'],
            "xgb_learning_rate": xgb_params['learning_rate'],
            "undersampling_ratio": undersample_ratio
        }
    }
    
    result_path = result_dir / f"exp21_{sampling_method}_results.json"
    with open(result_path, 'w') as f:
        json.dump(result_dict, f, indent=4)
    print(f"\nHasil evaluasi lengkap tersimpan ke: {result_path}")

    # Simpan array prediksi mentah Test set untuk visualisasi kurva kontinu (PR & ROC)
    pred_path = result_dir / f"exp21_{sampling_method}_predictions.npz"
    np.savez_compressed(
        pred_path,
        test_probs=test_probs.astype(np.float32),
        test_labels=test_labels.astype(np.int8)
    )
    print(f"Prediksi test set mentah tersimpan ke: {pred_path}")

    history_dict = {
        "experiment_id": "EXP21",
        "sampling_method": sampling_method,
        "gnn_epochs": epochs,
        "best_epoch": int(best_epoch),
        "best_val_auprc": float(best_val_auprc),
        "gnn_history": gnn_history,
        "gnn_step_history": gnn_step_history,
        "xgb_history": {
            "train_logloss": [float(x) for x in xgb_evals_result.get('validation_0', {}).get('logloss', [])],
            "val_logloss": [float(x) for x in xgb_evals_result.get('validation_1', {}).get('logloss', [])],
            "train_aucpr": [float(x) for x in xgb_evals_result.get('validation_0', {}).get('aucpr', [])],
            "val_aucpr": [float(x) for x in xgb_evals_result.get('validation_1', {}).get('aucpr', [])],
            "train_auc": [float(x) for x in xgb_evals_result.get('validation_0', {}).get('auc', [])],
            "val_auc": [float(x) for x in xgb_evals_result.get('validation_1', {}).get('auc', [])]
        }
    }
    history_path = result_dir / f"exp21_{sampling_method}_history.json"
    with open(history_path, 'w') as f:
        json.dump(history_dict, f, indent=4)
    print(f"Riwayat training per-epoch lengkap tersimpan ke: {history_path}")

if __name__ == '__main__':
    main()



In [ ]:
%%writefile run_all.py
import os
import subprocess
import sys
import yaml

def run_script(script_name, args=[]):
    print(f"\n{'='*70}")
    print(f"  >>> MENJALANKAN SCRIPT: {script_name} {' '.join(args)}")
    print(f"{'='*70}\n")
    
    command = [sys.executable, script_name] + args
    result = subprocess.run(command)
    
    if result.returncode != 0:
        print(f"\n[ERROR] Eksekusi {script_name} gagal dengan kode {result.returncode}!")
        sys.exit(result.returncode)
    else:
        print(f"\n[SUCCESS] {script_name} selesai dieksekusi.\n")

if __name__ == "__main__":
    print("="*70)
    print("  MEMULAI EKSPERIMEN 21: TEMPORAL WINDOW ALIGNMENT (GRAPHSAGE + XGBOOST)")
    print("="*70)
    
    # 1. Install dependencies jika di environment Kaggle/Colab
    if os.path.exists("/kaggle/input") or os.path.exists("/content"):
        print("\n[INFO] Menginstall dependencies (torch_geometric & pyg-lib)...")
        try:
            import torch
            pt_version = torch.__version__.split('+')[0]
            cuda_version = torch.version.cuda.replace('.', '') if torch.cuda.is_available() and torch.version.cuda else 'cpu'
            if cuda_version != 'cpu':
                url = f"https://data.pyg.org/whl/torch-{pt_version}+cu{cuda_version}.html"
                os.system(f"pip install torch_geometric pyg_lib torch_scatter torch_sparse -f {url} > /dev/null 2>&1")
            else:
                os.system("pip install torch_geometric > /dev/null 2>&1")
        except Exception as e:
            print(f"[WARNING] Fallback install torch_geometric: {e}")
            os.system("pip install torch_geometric > /dev/null 2>&1")

    # 2. Deteksi path konfigurasi
    if os.path.exists("/kaggle/input"):
        config_path = "config.kaggle.yaml"
        # Deteksi otomatis CSV di Kaggle
        csv_path = None
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'credit_card_transactions-ibm_v2.csv' in files:
                csv_path = os.path.join(root, 'credit_card_transactions-ibm_v2.csv')
                break
        if csv_path:
            print(f"[INFO] Dataset ditemukan di: {csv_path}")
            with open(config_path, 'r') as f:
                cfg = yaml.safe_load(f)
            cfg['data']['transactions'] = csv_path
            with open(config_path, 'w') as f:
                yaml.dump(cfg, f)
    elif os.path.exists("/content"):
        config_path = "config.colab.yaml"
    else:
        config_path = "config.reference.yaml"

    # 3. Jalankan eksperimen untuk metode sampling
    sampling_methods = ["uniform", "topology", "importance"]
    for method in sampling_methods:
        print(f"\n{'*'*70}")
        print(f"  >>> MULAI EKSPERIMEN DENGAN SAMPLING: {method.upper()}")
        print(f"{'*'*70}\n")
        run_script("main.py", ["--config", config_path, "--sampling", method])
        
    # 4. Generate Visualisasi Hasil (Chart 1 - 15)
    if os.path.exists("visualize_results.py"):
        print("\n" + "="*70)
        print("  >>> MENJALANKAN GENERASI VISUALISASI HASIL (CHART 1-17)")
        print("="*70 + "\n")
        try:
            with open(config_path, 'r') as f:
                cfg_loaded = yaml.safe_load(f)
            res_dir = cfg_loaded.get('output', {}).get('result_dir', 'result/exp21')
            plots_dir = os.path.join(res_dir, 'plots')
            run_script("visualize_results.py", ["--result-dir", res_dir, "--output-dir", plots_dir])
        except Exception as e:
            print(f"[WARNING] Gagal menjalankan visualisasi otomatis: {e}")

    print("\n" + "="*70)
    print("  🎉 SELURUH PROSES PELATIHAN, EVALUASI, DAN VISUALISASI EXP21 SELESAI! 🎉")
    print("="*70)




In [ ]:
!python run_all.py